# UWF Zeek – Windowed Graph Preprocessing (v2)

**Goal:** rebuild the UWF graph so it has enough *samples* to learn from, using **UWF Zeek conn logs only**.
Phase A collapsed 27M logs into 2,183 edges (one per IP pair). Here an edge becomes **(src IP, dst IP, time window)**.

**What is preserved from Phase A (same logic, same names where possible)**
- Same 6 UWF Zeek datasets (50 parquet files, ~27.11M rows) – schema is harmonised across years.
- Drop rows with `label_binary == 'Duplicate'` (→ 27,078,541 rows).
- `final_target` = Attack if `label_tactic` ∉ {`none`, NULL}, else Normal.
- `label_technique`: Normal → `none`; unknown Attack technique → `<Tactic>_unspecified`.
- `label_cve` dropped; `history` → `-`, `service` → `unknown`, `duration/orig_bytes/resp_bytes` → 0, `vlan` normalised.
- `ln(1+x)` scaling of duration/bytes/packets, proto one-hot (tcp/udp/icmp), top-5 service flags.
- IP → integer node id, sorted by IP string (ids 0..1175 identical to Phase A).
- Phase A's 8 edge features (`edge_weight`, `avg_log_*`, `total_tcp/udp/http/dns`) – kept as the `legacy` feature group.
- **Parity check (section 6)** re-creates the old whole-graph aggregation and compares with Phase A (1,176 nodes / 2,183 edges).

**What is new**
- Time windows (`WINDOW_MIN`, chosen from the diagnostics in section 5).
- Extra edge features: `conn_state` fractions, TCP-flag counts from `history`, port-group fractions, distinct ports, duration/rate.
- Per-window node features (degrees, volumes, failed-connection ratio, distinct ports, PageRank, clustering) – **no IP-identity one-hot**.
- **Time-based train/val/test split** (with a purge gap) instead of a random edge split; scaler fitted on train only.
- Label-sparsity masks (100/15/10/5/1 % of attack labels), nested and stratified by technique.
- Optional smoke test: a small GraphSAGE edge classifier + an ablation to check whether volume features act as a shortcut.

**Run order:** top to bottom. Sections 1–4 take the longest (download + ingest ≈ 10–25 min on Colab). After section 5, look at the table and set `CFG.WINDOW_MIN` before running section 7.

## 0. Setup

In [1]:
# [cell:install]
!pip -q install duckdb torch-geometric networkx scipy requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.0 MB/s eta 0:00:00


In [2]:
# [cell:config]
import os, re, sys, json, time, shutil, urllib.parse, warnings
from pathlib import Path
import numpy as np, pandas as pd, duckdb, requests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

class CFG:
    # ---- source (public UWF dataset server) ----
    BASE_URL = "https://datasets.uwf.edu/data/"
    DATASETS = ["UWF-ZeekData22", "UWF-ZeekDataFall22", "UWF-ZeekData24",
                "UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-1", "UWF-ZeekDataSum25-2"]
    EXPECTED_FILES = 50                      # Phase A merged 50 parquet files
    # ---- paths ----
    LOCAL = Path("/content") if IN_COLAB else Path("./work")
    RAW_DIR = LOCAL / "uwf_raw"
    DB_PATH = LOCAL / "uwf_work.duckdb"
    TMP_DIR = LOCAL / "duck_tmp"
    OUT_DIR = Path("/content/drive/MyDrive/298B/uwf_graph_v2") if IN_COLAB else Path("./out")
    DRIVE_RAW_BACKUP = OUT_DIR.parent / "uwf_raw_backup"   # raw parquet cache on Drive (skip re-download)
    BACKUP_RAW_TO_DRIVE = True
    DUCKDB_MEMORY = "8GB"
    REINGEST = False                          # True = rebuild raw_logs from parquet
    # ---- graph construction ----
    WINDOW_MIN = 15                           # decide after section 5
    LABEL_RULE = "any"                        # 'any' (Phase A: MAX) or 'majority' (attack_frac >= .5)
    # ---- split ----
    SPLIT_MODE = "chrono_within_period"       # 'chrono' | 'chrono_within_period'
    TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
    GAP_WINDOWS = 1                           # purge windows right before each boundary
    # ---- few-shot ----
    SPARSITY_LEVELS = [100, 15, 10, 5, 1]     # % of attack labels kept in TRAIN
    SEED = 42
    # ---- extras ----
    COMPUTE_GRAPH_METRICS = True              # PageRank + clustering per window (networkx)
    RUN_SMOKE_TEST = True
    SMOKE_EPOCHS = 12
    SMOKE_MAX_TRAIN_WINDOWS = 4000           # random subset of train windows for speed

for p in [CFG.RAW_DIR, CFG.TMP_DIR, CFG.OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

EXPECTED = {"raw_rows": 27_111_594, "clean_rows": 27_078_541, "nodes": 1176,
            "legacy_edges": 2183, "legacy_attack_edges": 1235, "legacy_normal_edges": 948}
REPORT = {"config": {k: str(v) for k, v in vars(CFG).items() if k.isupper()}}
print("Colab:", IN_COLAB, "| raw:", CFG.RAW_DIR, "| out:", CFG.OUT_DIR)

Mounted at /content/drive
Colab: True | raw: /content/uwf_raw | out: /content/drive/MyDrive/298B/uwf_graph_v2


## 1. Fetch the data
All six UWF Zeek datasets are public at `https://datasets.uwf.edu/data/` (`<dataset>/parquet/<week>/part-*.parquet`, ~1.2 GB total).
`UWF-HostData25` (host logs, ~4 GB) is on the same server but is **not** used here.
Files already on Drive (`DRIVE_RAW_BACKUP`) are restored first, so the download only happens once.

In [3]:
# [cell:fetch]
def hrefs(url):
    html = requests.get(url, timeout=60).text
    return re.findall(r'<a href="([^"?]+)"', html)

def remote_files():
    out = []
    for ds in CFG.DATASETS:
        pq_url = f"{CFG.BASE_URL}{ds}/parquet/"
        for wk in hrefs(pq_url):
            if not wk.endswith("/") or wk.startswith(("..", "_")):
                continue
            for f in hrefs(pq_url + wk):
                if f.startswith("part-") and f.endswith(".parquet"):
                    out.append({"dataset": ds, "week": urllib.parse.unquote(wk.strip("/")),
                                "url": pq_url + wk + f, "file": f})
    return out

def fetch(url, dest, retries=3):
    err = None
    for a in range(retries):
        try:
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                size = int(r.headers.get("content-length", 0))
                if dest.exists() and size and dest.stat().st_size == size:
                    return "cached"
                tmp = dest.with_suffix(".part")
                with open(tmp, "wb") as fh:
                    for ch in r.iter_content(1 << 20):
                        fh.write(ch)
                if size and tmp.stat().st_size != size:
                    raise IOError("size mismatch")
                tmp.rename(dest)
                return "downloaded"
        except Exception as e:
            err = e; time.sleep(2 * (a + 1))
    raise RuntimeError(f"failed {url}: {err}")

# restore from Drive cache if present
if CFG.DRIVE_RAW_BACKUP.exists():
    shutil.copytree(CFG.DRIVE_RAW_BACKUP, CFG.RAW_DIR, dirs_exist_ok=True)
    print("restored cache from", CFG.DRIVE_RAW_BACKUP)

manifest = remote_files()
print(f"{len(manifest)} parquet files listed on server")
for m in manifest:
    d = CFG.RAW_DIR / m["dataset"] / re.sub(r"[^0-9A-Za-z]+", "_", m["week"])
    d.mkdir(parents=True, exist_ok=True)
    m["path"] = str(d / m["file"])
    m["status"] = fetch(m["url"], Path(m["path"]))
    m["mb"] = round(Path(m["path"]).stat().st_size / 1e6, 2)

json.dump(manifest, open(CFG.RAW_DIR / "manifest.json", "w"), indent=1)
man = pd.DataFrame(manifest)
print(man.groupby("dataset").agg(files=("file", "count"), MB=("mb", "sum")).round(1))
print("TOTAL files:", len(man), "| MB:", round(man.mb.sum(), 1), "| downloaded now:", (man.status == "downloaded").sum())
assert len(man) == CFG.EXPECTED_FILES, f"expected {CFG.EXPECTED_FILES} files (Phase A), found {len(man)} - check the server listing"

if CFG.BACKUP_RAW_TO_DRIVE and IN_COLAB:
    shutil.copytree(CFG.RAW_DIR, CFG.DRIVE_RAW_BACKUP, dirs_exist_ok=True)
    print("raw cache backed up to", CFG.DRIVE_RAW_BACKUP)

50 parquet files listed on server
                      files     MB
dataset                           
UWF-ZeekData22            8  568.8
UWF-ZeekData24            7  140.2
UWF-ZeekDataFall22       13   44.9
UWF-ZeekDataFall24-2      9  209.0
UWF-ZeekDataSum25-1       9   87.8
UWF-ZeekDataSum25-2       4  131.0
TOTAL files: 50 | MB: 1181.7 | downloaded now: 50
raw cache backed up to /content/drive/MyDrive/298B/uwf_raw_backup


## 2. Ingest into DuckDB (schema harmonised per file)
Columns differ by year (22–27 columns, int32 vs int64, `vlan`/`label_technique`/`label_binary`/`label_cve` missing in older files).
Each file is cast to one target schema (the Phase A `network_logs` table plus `source_dataset`); missing columns become NULL.

In [4]:
# [cell:ingest]
con = duckdb.connect(str(CFG.DB_PATH))
con.execute(f"SET memory_limit='{CFG.DUCKDB_MEMORY}'")
con.execute(f"SET temp_directory='{CFG.TMP_DIR}'")

TARGET = [("resp_pkts","BIGINT"),("service","VARCHAR"),("orig_ip_bytes","BIGINT"),("local_resp","BOOLEAN"),
          ("missed_bytes","BIGINT"),("proto","VARCHAR"),("duration","DOUBLE"),("conn_state","VARCHAR"),
          ("dest_ip_zeek","VARCHAR"),("orig_pkts","BIGINT"),("community_id","VARCHAR"),("resp_ip_bytes","BIGINT"),
          ("dest_port_zeek","INTEGER"),("orig_bytes","BIGINT"),("local_orig","BOOLEAN"),("datetime","TIMESTAMP"),
          ("history","VARCHAR"),("resp_bytes","BIGINT"),("uid","VARCHAR"),("src_port_zeek","INTEGER"),
          ("ts","DOUBLE"),("src_ip_zeek","VARCHAR"),("label_tactic","VARCHAR"),("label_technique","VARCHAR"),
          ("label_cve","VARCHAR"),("label_binary","VARCHAR"),("vlan","VARCHAR")]

have = con.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_name='raw_logs'").fetchone()[0]
n_have = con.execute("SELECT COUNT(*) FROM raw_logs").fetchone()[0] if have else 0
if n_have and not CFG.REINGEST:
    print(f"raw_logs already has {n_have:,} rows (set CFG.REINGEST=True to rebuild)")
else:
    con.execute("DROP TABLE IF EXISTS raw_logs")
    ddl = ", ".join(f"{c} {t}" for c, t in TARGET) + ", source_dataset VARCHAR, source_period VARCHAR"
    con.execute(f"CREATE TABLE raw_logs ({ddl})")
    manifest = json.load(open(CFG.RAW_DIR / "manifest.json"))
    t0 = time.time()
    for i, m in enumerate(manifest, 1):
        p = m["path"].replace("'", "''")
        cols = {r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").fetchall()}
        sel = [(f"CAST({c} AS {t}) AS {c}" if c in cols else f"CAST(NULL AS {t}) AS {c}") for c, t in TARGET]
        sel += [f"'{m['dataset']}' AS source_dataset", f"'{m['week']}' AS source_period"]
        con.execute(f"INSERT INTO raw_logs SELECT {', '.join(sel)} FROM read_parquet('{p}')")
        if i % 10 == 0 or i == len(manifest):
            print(f"  {i}/{len(manifest)} files  ({time.time()-t0:.0f}s)")

n_raw = con.execute("SELECT COUNT(*) FROM raw_logs").fetchone()[0]
print(f"raw_logs rows: {n_raw:,}")
print(con.execute('''SELECT source_dataset, COUNT(*) n_logs, MIN(datetime) first_ts, MAX(datetime) last_ts
                     FROM raw_logs GROUP BY 1 ORDER BY 3''').df())
FULL_DATA = (n_raw >= 0.99 * EXPECTED["raw_rows"])

def parity(name, actual, expected):
    if not FULL_DATA:
        print(f"  [skip] {name}: {actual:,} (partial data)"); return None
    ok = (actual == expected)
    print(f"  [{'PASS' if ok else 'DIFF'}] {name}: {actual:,} (Phase A: {expected:,})")
    return ok
REPORT["parity"] = {"raw_rows": parity("raw rows", n_raw, EXPECTED["raw_rows"])}

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  40/50 files  (137s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  50/50 files  (154s)
raw_logs rows: 27,111,594
         source_dataset    n_logs                   first_ts                    last_ts
0        UWF-ZeekData22  18562468 2021-12-17 07:00:45.213000 2022-02-19 19:16:36.163000
1    UWF-ZeekDataFall22    700340 2021-12-17 13:00:45.251239 2022-10-23 22:07:45.793000
2        UWF-ZeekData24   1916757 2024-02-28 04:00:37.805000 2024-11-05 23:59:48.335452
3  UWF-ZeekDataFall24-2   2909841 2024-09-21 21:06:52.207000 2024-12-09 23:59:51.394266
4   UWF-ZeekDataSum25-1   1153509 2025-05-23 00:00:00.895212 2025-07-31 14:56:10.005000
5   UWF-ZeekDataSum25-2   1868679 2025-05-23 04:37:17.535901 2025-06-09 20:12:33.072000
  [PASS] raw rows: 27,111,594 (Phase A: 27,111,594)


## 3. Phase A cleaning (reproduced in one SQL statement)
Same rules as `2. EDA-Data Prep.ipynb`: drop `Duplicate`, derive `final_target`, fix `label_technique`, drop `label_cve`, fill nulls, normalise `vlan`.

In [8]:
# [cell:clean]
con.execute(r'''
CREATE OR REPLACE TABLE logs_clean AS
WITH t AS (
  SELECT *,
    COALESCE(NULLIF(label_binary,'nan'),'unknown')    AS lb,
    COALESCE(NULLIF(label_technique,'nan'),'unknown') AS lt,
    CASE WHEN label_tactic IS NOT NULL AND label_tactic <> 'none' THEN 'Attack' ELSE 'Normal' END AS final_target
  FROM raw_logs
)
SELECT
  resp_pkts, COALESCE(service,'unknown') AS service, orig_ip_bytes, local_resp, missed_bytes, proto,
  COALESCE(duration,0) AS duration, conn_state, dest_ip_zeek, orig_pkts, community_id, resp_ip_bytes,
  dest_port_zeek, COALESCE(orig_bytes,0) AS orig_bytes, local_orig, datetime,
  COALESCE(history,'-') AS history, COALESCE(resp_bytes,0) AS resp_bytes, uid, src_port_zeek, ts, src_ip_zeek,
  label_tactic,
  CASE WHEN final_target = 'Normal' THEN 'none'
       WHEN lt = 'unknown' THEN label_tactic || '_unspecified'
       ELSE lt END AS label_technique,
  lb AS label_binary,
  COALESCE(CAST(TRY_CAST(TRY_CAST(NULLIF(NULLIF(vlan,'unknown'),'nan') AS DOUBLE) AS INTEGER) AS VARCHAR), 'none') AS vlan,
  final_target, source_dataset, source_period
FROM t
WHERE lb <> 'Duplicate' AND src_ip_zeek IS NOT NULL AND dest_ip_zeek IS NOT NULL AND ts IS NOT NULL
''')
n_clean = con.execute("SELECT COUNT(*) FROM logs_clean").fetchone()[0]
print(f"logs_clean rows: {n_clean:,}  (dropped {n_raw-n_clean:,} 'Duplicate' rows)")
REPORT["parity"]["clean_rows"] = parity("clean rows", n_clean, EXPECTED["clean_rows"])

print("\nfinal_target by dataset (log level):")
print(con.execute('''SELECT source_dataset, COUNT(*) logs,
        SUM((final_target='Attack')::INT) attack_logs,
        ROUND(100.0*AVG((final_target='Attack')::INT),2) attack_pct
      FROM logs_clean GROUP BY 1 ORDER BY 1''').df())
print("\nlabel_technique (top 15):")
print(con.execute('''SELECT label_technique, COUNT(*) logs FROM logs_clean
                     GROUP BY 1 ORDER BY 2 DESC LIMIT 15''').df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

logs_clean rows: 27,078,541  (dropped 33,053 'Duplicate' rows)
  [PASS] clean rows: 27,078,541 (Phase A: 27,078,541)

final_target by dataset (log level):
         source_dataset      logs  attack_logs  attack_pct
0        UWF-ZeekData22  18562468    9280869.0       50.00
1        UWF-ZeekData24   1898613     940504.0       49.54
2    UWF-ZeekDataFall22    697272     346933.0       49.76
3  UWF-ZeekDataFall24-2   2901532    1446262.0       49.84
4   UWF-ZeekDataSum25-1   1149977     573218.0       49.85
5   UWF-ZeekDataSum25-2   1868679     934453.0       50.01

label_technique (top 15):
               label_technique      logs
0                         none  13556302
1   Reconnaissance_unspecified   9278722
2                        T1046   1779059
3                        T1595   1197737
4                        T1110    871188
5                        T1587    275471
6                        T1018     30774
7                        T1592     21405
8                        T1590     2

## 4. Log-level feature table
Keeps Phase A's `log_*` / `proto_*` / `srv_*` columns and adds: `conn_state` flags, TCP-flag counts from `history`
(Zeek: upper-case = originator, lower-case = responder), and destination-port groups.

In [6]:
# [cell:features_view]
def _cnt(chars):
    return " + ".join(f"(LENGTH(history) - LENGTH(REPLACE(history,'{c}','')))" for c in chars)

con.execute(f'''
CREATE OR REPLACE TABLE logs_feat AS
SELECT
  source_dataset, source_period, src_ip_zeek AS src_ip, dest_ip_zeek AS dst_ip, ts,
  dest_port_zeek, src_port_zeek, label_tactic, label_technique, final_target,
  CASE WHEN final_target='Attack' THEN 1 ELSE 0 END AS attack,
  -- ---- Phase A features ----
  LN(1 + GREATEST(duration,0))   AS log_duration,
  LN(1 + GREATEST(orig_bytes,0)) AS log_orig_bytes,
  LN(1 + GREATEST(resp_bytes,0)) AS log_resp_bytes,
  LN(1 + GREATEST(orig_pkts,0))  AS log_orig_pkts,
  LN(1 + GREATEST(resp_pkts,0))  AS log_resp_pkts,
  CASE WHEN proto='tcp'  THEN 1 ELSE 0 END AS proto_tcp,
  CASE WHEN proto='udp'  THEN 1 ELSE 0 END AS proto_udp,
  CASE WHEN proto='icmp' THEN 1 ELSE 0 END AS proto_icmp,
  CASE WHEN service='http'    THEN 1 ELSE 0 END AS srv_http,
  CASE WHEN service='dns'     THEN 1 ELSE 0 END AS srv_dns,
  CASE WHEN service='ssl'     THEN 1 ELSE 0 END AS srv_ssl,
  CASE WHEN service='ssh'     THEN 1 ELSE 0 END AS srv_ssh,
  CASE WHEN service='unknown' THEN 1 ELSE 0 END AS srv_unknown,
  -- ---- new: connection state ----
  CASE WHEN conn_state='SF' THEN 1 ELSE 0 END AS cs_sf,
  CASE WHEN conn_state='S0' THEN 1 ELSE 0 END AS cs_s0,
  CASE WHEN conn_state='REJ' THEN 1 ELSE 0 END AS cs_rej,
  CASE WHEN conn_state IN ('RSTO','RSTR','RSTOS0','RSTRH') THEN 1 ELSE 0 END AS cs_rst,
  CASE WHEN conn_state='OTH' THEN 1 ELSE 0 END AS cs_oth,
  CASE WHEN conn_state IN ('S1','S2','S3','SH','SHR') THEN 1 ELSE 0 END AS cs_partial,
  -- ---- new: TCP flag counts from history ----
  CASE WHEN history='-' THEN 0 ELSE LENGTH(history) END AS h_len,
  {_cnt('S')}   AS h_S,
  {_cnt('h')}   AS h_h,
  {_cnt('Aa')}  AS h_ack,
  {_cnt('Dd')}  AS h_data,
  {_cnt('Ff')}  AS h_fin,
  {_cnt('Rr')}  AS h_rst,
  {_cnt('Tt')}  AS h_retx,
  -- ---- new: destination port groups (mutually exclusive) ----
  CASE WHEN dest_port_zeek IN (80,443,8080,8443,8000) THEN 1 ELSE 0 END AS pg_web,
  CASE WHEN dest_port_zeek = 22 THEN 1 ELSE 0 END AS pg_ssh,
  CASE WHEN dest_port_zeek IN (445,139,3389) THEN 1 ELSE 0 END AS pg_smb_rdp,
  CASE WHEN dest_port_zeek = 53 THEN 1 ELSE 0 END AS pg_dns,
  CASE WHEN dest_port_zeek < 1024 AND dest_port_zeek NOT IN (80,443,22,445,139,53) THEN 1 ELSE 0 END AS pg_other_wk,
  CASE WHEN dest_port_zeek >= 1024 AND dest_port_zeek < 49152 AND dest_port_zeek NOT IN (8080,8443,8000,3389) THEN 1 ELSE 0 END AS pg_registered,
  CASE WHEN dest_port_zeek >= 49152 THEN 1 ELSE 0 END AS pg_ephemeral
FROM logs_clean
''')
print(con.execute("SELECT * FROM logs_feat LIMIT 3").df().T.head(45))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                                       0                        1                         2
source_dataset            UWF-ZeekData22           UWF-ZeekData22            UWF-ZeekData22
source_period    2021-12-12 - 2021-12-19  2021-12-12 - 2021-12-19   2021-12-12 - 2021-12-19
src_ip                     143.88.255.50             143.88.255.1  fe80::250:56ff:fe9e:da15
dst_ip                        143.88.0.1            143.88.255.10                   ff02::1
ts                     1639746045.896897        1639746047.630312         1639746045.879347
dest_port_zeek                         0                        1                       133
src_port_zeek                          8                        3                       134
label_tactic                        none                     none                      none
label_technique                     none                     none                      none
final_target                      Normal                   Normal               

## 5. Window-size diagnostics – *decide `WINDOW_MIN` here*
For each candidate window: how many edges (= training samples), how many attack edges, how many edges mix attack and normal logs,
and how many labelled attack edges would survive at 1 % label availability. Windows are built **per dataset** so concurrent captures are never merged.

**Paste this table (and the per-dataset table) back to me before choosing.**

In [7]:
# [cell:window_diag]
def window_diag(W):
    ws = W * 60
    con.execute(f'''CREATE OR REPLACE TEMP TABLE _e AS
        SELECT source_dataset ds, CAST(FLOOR(ts/{ws}) AS BIGINT) w, src_ip, dst_ip,
               MAX(attack) a, AVG(attack) af, COUNT(*) n
        FROM logs_feat GROUP BY 1,2,3,4''')
    s = con.execute('''SELECT COUNT(*) edges, COUNT(DISTINCT ds||':'||w) windows,
                       SUM(a) attack_edges, SUM((af>0 AND af<1)::INT) mixed_edges FROM _e''').fetchone()
    per_w = con.execute('''SELECT ds, w, COUNT(*) n_edges, SUM(a) n_att FROM _e GROUP BY 1,2''').df()
    nodes = con.execute('''SELECT ds, w, COUNT(DISTINCT ip) nodes FROM
                           (SELECT ds,w,src_ip ip FROM _e UNION ALL SELECT ds,w,dst_ip FROM _e) GROUP BY 1,2''').df()
    edges, windows, att, mixed = s
    return {"window_min": W, "edges": edges, "windows": windows, "attack_edges": int(att),
            "normal_edges": int(edges - att), "attack_%": round(100*att/edges, 1),
            "mixed_edges_%": round(100*mixed/edges, 2),
            "edges/window p50": int(per_w.n_edges.median()), "edges/window p90": int(per_w.n_edges.quantile(.9)),
            "nodes/window p50": int(nodes.nodes.median()), "nodes/window p90": int(nodes.nodes.quantile(.9)),
            "windows_with_attack_%": round(100*(per_w.n_att > 0).mean(), 1),
            "train_attack_edges@100%": int(att*CFG.TRAIN_FRAC),
            "train_attack_edges@1%": int(att*CFG.TRAIN_FRAC*0.01)}

diag = pd.DataFrame([window_diag(W) for W in (5, 15, 60)]).set_index("window_min")
print(diag.T)
REPORT["window_diagnostics"] = diag.reset_index().to_dict("records")

print("\nCoverage per dataset (days with data, attack share of logs):")
print(con.execute('''SELECT source_dataset, COUNT(DISTINCT CAST(datetime AS DATE)) active_days,
        MIN(CAST(datetime AS DATE)) first_day, MAX(CAST(datetime AS DATE)) last_day,
        ROUND(100*AVG((final_target='Attack')::INT),2) attack_pct_logs
        FROM logs_clean GROUP BY 1 ORDER BY 3''').df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

window_min                     5         15        60
edges                    726307.0  438683.0  224097.0
windows                   33607.0   11444.0    2996.0
attack_edges              54650.0   51709.0   43485.0
normal_edges             671657.0  386974.0  180612.0
attack_%                      7.5      11.8      19.4
mixed_edges_%                 0.0       0.0       0.0
edges/window p50             16.0      36.0      74.0
edges/window p90             44.0      73.0     142.0
nodes/window p50             23.0      39.0      60.0
nodes/window p90             55.0      76.0     106.0
windows_with_attack_%        17.3      19.0      22.6
train_attack_edges@100%   38255.0   36196.0   30439.0
train_attack_edges@1%       382.0     361.0     304.0

Coverage per dataset (days with data, attack share of logs):
         source_dataset  active_days  first_day   last_day  attack_pct_logs
0        UWF-ZeekData22           37 2021-12-17 2022-02-19            50.00
1    UWF-ZeekDataFall22       

## 6. Phase A parity check (whole-graph aggregation)
Re-creates Phase A's `graph_edges` / `gnn_edge_attr`: one edge per (src, dst) over **all** time. With the full data this must give
**1,176 nodes, 2,183 edges (1,235 attack / 948 normal)**. If it does, everything upstream (download, harmonisation, cleaning, labels) matches Phase A.

In [9]:
# [cell:legacy_parity]
con.execute('''
CREATE OR REPLACE TABLE legacy_edges AS
SELECT src_ip, dst_ip,
       LN(1+COUNT(*)) AS edge_weight, AVG(log_duration) AS avg_log_duration,
       AVG(log_orig_bytes) AS avg_log_orig_bytes, AVG(log_resp_bytes) AS avg_log_resp_bytes,
       SUM(proto_tcp) AS total_tcp, SUM(proto_udp) AS total_udp, SUM(srv_http) AS total_http, SUM(srv_dns) AS total_dns,
       MAX(attack) AS is_attack_edge
FROM logs_feat GROUP BY 1,2''')
n_nodes = con.execute("SELECT COUNT(*) FROM (SELECT src_ip ip FROM logs_feat UNION SELECT dst_ip FROM logs_feat)").fetchone()[0]
le = con.execute("SELECT COUNT(*), SUM(is_attack_edge) FROM legacy_edges").fetchone()
REPORT["parity"]["nodes"]        = parity("unique IPs (nodes)", n_nodes, EXPECTED["nodes"])
REPORT["parity"]["legacy_edges"] = parity("legacy edges", le[0], EXPECTED["legacy_edges"])
REPORT["parity"]["legacy_attack_edges"] = parity("legacy attack edges", int(le[1]), EXPECTED["legacy_attack_edges"])
REPORT["parity"]["legacy_normal_edges"] = parity("legacy normal edges", int(le[0]-le[1]), EXPECTED["legacy_normal_edges"])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  [PASS] unique IPs (nodes): 1,176 (Phase A: 1,176)
  [PASS] legacy edges: 2,183 (Phase A: 2,183)
  [PASS] legacy attack edges: 1,235 (Phase A: 1,235)
  [PASS] legacy normal edges: 948 (Phase A: 948)


## 7. Node mapping + windowed edge table
Set `CFG.WINDOW_MIN` (cell 0) from section 5 before running. Node ids are IPs sorted alphabetically (identical to Phase A's `node_mapping`).
Edge label = Phase A's rule (`MAX(attack)`) by default; `attack_frac`, `n_attack_logs`, the dominant technique/tactic and the number of distinct techniques are kept so mixed edges can be audited.

In [10]:
# [cell:nodes_and_edges]
W = CFG.WINDOW_MIN; WS = W * 60
con.execute('''
CREATE OR REPLACE TABLE node_mapping AS
SELECT ip, ROW_NUMBER() OVER (ORDER BY ip) - 1 AS node_id
FROM (SELECT src_ip ip FROM logs_feat UNION SELECT dst_ip FROM logs_feat)''')
con.execute(r'''
CREATE OR REPLACE TABLE node_static AS
SELECT node_id, ip,
  CASE WHEN regexp_matches(ip, '^(10\.|192\.168\.|172\.(1[6-9]|2[0-9]|3[01])\.|143\.88\.)') THEN 1 ELSE 0 END AS is_internal,
  CASE WHEN ip LIKE '%:%' THEN 1 ELSE 0 END AS is_ipv6,
  CASE WHEN ip IN ('0.0.0.0','255.255.255.255','127.0.0.1') OR ip LIKE 'ff%:%' OR ip LIKE 'fe80:%'
         OR regexp_matches(ip, '^(22[4-9]|23[0-9])\.') THEN 1 ELSE 0 END AS is_special,
  CASE WHEN regexp_matches(ip, '^143\.88\.[0-9]+\.') THEN CAST(regexp_extract(ip, '^143\.88\.([0-9]+)\.', 1) AS INTEGER) ELSE -1 END AS subnet_id
FROM node_mapping''')
print("nodes:", con.execute("SELECT COUNT(*) FROM node_mapping").fetchone()[0])
print(con.execute("SELECT SUM(is_internal) internal, SUM(is_ipv6) ipv6, SUM(is_special) special FROM node_static").df())

con.execute(f'''
CREATE OR REPLACE TABLE edges_raw AS
SELECT source_dataset, CAST(FLOOR(ts/{WS}) AS BIGINT) AS w, src_ip, dst_ip,
  MIN(ts) AS ts_first, MAX(ts) AS ts_last, COUNT(*) AS n_logs,
  LN(1+COUNT(*)) AS edge_weight,
  AVG(log_duration) AS avg_log_duration, AVG(log_orig_bytes) AS avg_log_orig_bytes, AVG(log_resp_bytes) AS avg_log_resp_bytes,
  AVG(log_orig_pkts) AS avg_log_orig_pkts, AVG(log_resp_pkts) AS avg_log_resp_pkts,
  SUM(proto_tcp) AS total_tcp, SUM(proto_udp) AS total_udp, SUM(proto_icmp) AS total_icmp,
  SUM(srv_http) AS total_http, SUM(srv_dns) AS total_dns, SUM(srv_ssl) AS total_ssl,
  SUM(srv_ssh) AS total_ssh, SUM(srv_unknown) AS total_unknown,
  AVG(cs_sf) AS f_cs_sf, AVG(cs_s0) AS f_cs_s0, AVG(cs_rej) AS f_cs_rej, AVG(cs_rst) AS f_cs_rst,
  AVG(cs_oth) AS f_cs_oth, AVG(cs_partial) AS f_cs_partial,
  AVG(h_len) AS m_h_len, AVG(h_S) AS m_h_S, AVG(h_h) AS m_h_h, AVG(h_ack) AS m_h_ack,
  AVG(h_data) AS m_h_data, AVG(h_fin) AS m_h_fin, AVG(h_rst) AS m_h_rst, AVG(h_retx) AS m_h_retx,
  AVG(pg_web) AS f_pg_web, AVG(pg_ssh) AS f_pg_ssh, AVG(pg_smb_rdp) AS f_pg_smb_rdp, AVG(pg_dns) AS f_pg_dns,
  AVG(pg_other_wk) AS f_pg_other_wk, AVG(pg_registered) AS f_pg_registered, AVG(pg_ephemeral) AS f_pg_ephemeral,
  COUNT(DISTINCT dest_port_zeek) AS n_dst_ports, COUNT(DISTINCT src_port_zeek) AS n_src_ports,
  LN(1 + (MAX(ts)-MIN(ts))) AS log_span, LN(1 + COUNT(*)/(1 + MAX(ts)-MIN(ts))) AS log_rate,
  MAX(attack) AS y_any, AVG(attack) AS attack_frac, SUM(attack) AS n_attack_logs,
  MODE(label_technique) FILTER (WHERE attack=1) AS tech_mode,
  MODE(label_tactic)    FILTER (WHERE attack=1) AS tactic_mode,
  COUNT(DISTINCT label_technique) FILTER (WHERE attack=1) AS n_techniques
FROM logs_feat GROUP BY 1,2,3,4''')

con.execute(f'''
CREATE OR REPLACE TABLE windows AS
SELECT CAST(ROW_NUMBER() OVER (ORDER BY w, source_dataset) - 1 AS INTEGER) AS win_id,
       source_dataset, w, w*{WS} AS win_start_ts, COUNT(*) AS n_edges, SUM(y_any) AS n_attack_edges
FROM edges_raw GROUP BY source_dataset, w''')

rule = CFG.LABEL_RULE
con.execute(f'''
CREATE OR REPLACE TABLE edges_win AS
SELECT wd.win_id, s.node_id AS src_id, d.node_id AS dst_id,
  e.* EXCLUDE (source_dataset, w, src_ip, dst_ip),
  CASE WHEN '{rule}'='any' THEN e.y_any ELSE CAST(e.attack_frac >= 0.5 AS INTEGER) END AS y
FROM edges_raw e
JOIN windows wd ON wd.source_dataset = e.source_dataset AND wd.w = e.w
JOIN node_mapping s ON s.ip = e.src_ip
JOIN node_mapping d ON d.ip = e.dst_ip
ORDER BY wd.win_id, s.node_id, d.node_id''')

n_e, n_w, n_a = con.execute("SELECT COUNT(*), COUNT(DISTINCT win_id), SUM(y) FROM edges_win").fetchone()
print(f"window={W} min | windows: {n_w:,} | edges: {n_e:,} | attack edges: {int(n_a):,} ({100*n_a/n_e:.1f}%)")
print(con.execute('''SELECT tech_mode, COUNT(*) edges FROM edges_win WHERE y=1
                     GROUP BY 1 ORDER BY 2 DESC LIMIT 20''').df())
REPORT["edges"] = {"window_min": W, "windows": int(n_w), "edges": int(n_e), "attack_edges": int(n_a)}

nodes: 1176
   internal  ipv6  special
0    1082.0  64.0     56.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

window=15 min | windows: 11,444 | edges: 438,683 | attack edges: 51,709 (11.8%)
                           tech_mode  edges
0                              T1595  36761
1                              T1110   5186
2                              T1078   4239
3                              T1190   3696
4                              T1046    542
5         Reconnaissance_unspecified    290
6                              T1018    273
7                              T1548    264
8                              T1592    230
9                              T1048    119
10                             T1587     23
11                             T1589     18
12                             T1210     17
13                             T1016     13
14     Credential Access_unspecified      6
15                             T1071      4
16                             T1590      3
17  Privilege Escalation_unspecified      3
18          Exfiltration_unspecified      3
19                             T1059    

In [11]:
display(con.execute("SELECT * FROM legacy_edges LIMIT 5").df().T); display(con.execute("SELECT * FROM edges_win WHERE y=1 LIMIT 5").df().T)


,0,1,2,3,4
src_ip,143.88.255.10,143.88.1.51,143.88.1.53,143.88.0.2,143.88.0.2
dst_ip,10.0.10.1,143.88.1.1,143.88.1.1,198.97.190.53,199.9.14.201
edge_weight,15.761394,12.708949,11.406553,8.495152,8.47762
avg_log_duration,0.000016,1.786619,0.000475,0.0,0.0
avg_log_orig_bytes,4.502559,5.191258,5.37659,0.0,0.0
avg_log_resp_bytes,0.0,3.897663,4.127154,0.0,0.0
total_tcp,0.0,0.0,0.0,0.0,0.0
total_udp,6999813.0,330693.0,89908.0,4890.0,4805.0
total_http,0.0,0.0,0.0,0.0,0.0
total_dns,6999813.0,329629.0,88991.0,4890.0,4805.0


,0,1,2,3,4
win_id,5090,5091,5092,5093,5094
src_id,511,511,511,511,511
dst_id,508,508,508,508,508
ts_first,1644465509.979958,1644465600.089253,1644466501.064527,1644467402.06471,1644468303.093402
ts_last,1644465590.08073,1644466491.054143,1644467392.054299,1644468293.084532,1644469194.09166
n_logs,4608,72704,73472,71424,71680
edge_weight,8.435766,11.194165,11.204673,11.176403,11.179981
avg_log_duration,0.897357,0.595691,0.612013,0.613392,0.606963
avg_log_orig_bytes,4.622323,2.995446,2.986431,3.069978,3.055757
avg_log_resp_bytes,2.437599,2.665311,2.808597,2.667507,2.731145


## 8. Per-window node features
Computed **only from the window itself** (no future information, no labels). Structural: in/out degree, volumes, failed-connection ratio,
distinct ports contacted/received, in/out imbalance, PageRank, clustering. Static: `is_internal`, `is_ipv6`, `is_special`.

In [12]:
# [cell:node_features]
con.execute(f'''
CREATE OR REPLACE TABLE node_win AS
WITH base AS (
  SELECT win_id, src_id AS node_id FROM edges_win UNION SELECT win_id, dst_id FROM edges_win),
o AS (
  SELECT win_id, src_id AS node_id, COUNT(*) AS out_deg, SUM(n_logs) AS out_logs,
         SUM(n_logs*avg_log_orig_bytes)/SUM(n_logs) AS out_lob, SUM(n_logs*avg_log_resp_bytes)/SUM(n_logs) AS out_lrb,
         SUM(n_logs*(f_cs_s0+f_cs_rej+f_cs_rst))/SUM(n_logs) AS out_fail, SUM(n_logs*avg_log_duration)/SUM(n_logs) AS out_dur
  FROM edges_win GROUP BY 1,2),
i AS (
  SELECT win_id, dst_id AS node_id, COUNT(*) AS in_deg, SUM(n_logs) AS in_logs,
         SUM(n_logs*avg_log_orig_bytes)/SUM(n_logs) AS in_lob, SUM(n_logs*avg_log_resp_bytes)/SUM(n_logs) AS in_lrb,
         SUM(n_logs*(f_cs_s0+f_cs_rej+f_cs_rst))/SUM(n_logs) AS in_fail, SUM(n_logs*avg_log_duration)/SUM(n_logs) AS in_dur
  FROM edges_win GROUP BY 1,2),
op AS (
  SELECT wd.win_id, n.node_id, COUNT(DISTINCT l.dest_port_zeek) AS out_ports
  FROM logs_feat l
  JOIN windows wd ON wd.source_dataset = l.source_dataset AND wd.w = CAST(FLOOR(l.ts/{WS}) AS BIGINT)
  JOIN node_mapping n ON n.ip = l.src_ip GROUP BY 1,2),
ip AS (
  SELECT wd.win_id, n.node_id, COUNT(DISTINCT l.dest_port_zeek) AS in_ports
  FROM logs_feat l
  JOIN windows wd ON wd.source_dataset = l.source_dataset AND wd.w = CAST(FLOOR(l.ts/{WS}) AS BIGINT)
  JOIN node_mapping n ON n.ip = l.dst_ip GROUP BY 1,2)
SELECT b.win_id, b.node_id,
  COALESCE(o.out_deg,0) AS out_deg, COALESCE(i.in_deg,0) AS in_deg,
  COALESCE(o.out_logs,0) AS out_logs, COALESCE(i.in_logs,0) AS in_logs,
  COALESCE(o.out_lob,0) AS out_lob, COALESCE(o.out_lrb,0) AS out_lrb, COALESCE(i.in_lob,0) AS in_lob, COALESCE(i.in_lrb,0) AS in_lrb,
  COALESCE(o.out_fail,0) AS out_fail, COALESCE(i.in_fail,0) AS in_fail,
  COALESCE(o.out_dur,0) AS out_dur, COALESCE(i.in_dur,0) AS in_dur,
  COALESCE(op.out_ports,0) AS out_ports, COALESCE(ip.in_ports,0) AS in_ports
FROM base b
LEFT JOIN o USING (win_id, node_id) LEFT JOIN i USING (win_id, node_id)
LEFT JOIN op USING (win_id, node_id) LEFT JOIN ip USING (win_id, node_id)
ORDER BY b.win_id, b.node_id''')

node_win = con.execute('''SELECT nw.*, ns.is_internal, ns.is_ipv6, ns.is_special
                          FROM node_win nw JOIN node_static ns USING (node_id)
                          ORDER BY win_id, node_id''').df()
for c in ["out_deg","in_deg","out_logs","in_logs","out_ports","in_ports"]:
    node_win[c + "_l"] = np.log1p(node_win[c])
tot = node_win.out_logs + node_win.in_logs
node_win["io_ratio"] = np.where(tot > 0, (node_win.out_logs - node_win.in_logs) / np.maximum(tot, 1), 0.0)
print("node-window rows:", f"{len(node_win):,}", "| windows:", node_win.win_id.nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

node-window rows: 467,856 | windows: 11444


In [13]:
# [cell:graph_metrics]
node_win["pagerank"] = 0.0; node_win["clustering"] = 0.0
if CFG.COMPUTE_GRAPH_METRICS:
    import networkx as nx
    ed = con.execute("SELECT win_id, src_id, dst_id FROM edges_win ORDER BY win_id").df()
    wid = ed.win_id.values
    bounds = np.r_[0, np.flatnonzero(np.diff(wid)) + 1, len(ed)]
    nw_win = node_win.win_id.values
    nw_bounds = np.r_[0, np.flatnonzero(np.diff(nw_win)) + 1, len(node_win)]
    assert len(bounds) == len(nw_bounds), "edge/node window mismatch"
    pr_out = np.zeros(len(node_win)); cl_out = np.zeros(len(node_win))
    t0 = time.time()
    for k in range(len(bounds) - 1):
        a, b = bounds[k], bounds[k+1]
        s, d = ed.src_id.values[a:b], ed.dst_id.values[a:b]
        G = nx.DiGraph(); G.add_edges_from(zip(s.tolist(), d.tolist()))
        pr = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-6)
        cl = nx.clustering(G.to_undirected())
        na, nb = nw_bounds[k], nw_bounds[k+1]
        ids = node_win.node_id.values[na:nb]
        n_act = len(ids)
        pr_out[na:nb] = [pr.get(int(i), 0.0) * n_act for i in ids]     # mean ≈ 1 within a window
        cl_out[na:nb] = [cl.get(int(i), 0.0) for i in ids]
        if k == 49:
            est = (time.time() - t0) / 50 * (len(bounds) - 1)
            print(f"  est. total {est/60:.1f} min for {len(bounds)-1:,} windows (set COMPUTE_GRAPH_METRICS=False to skip)")
    node_win["pagerank"] = pr_out; node_win["clustering"] = cl_out
    print(f"graph metrics done in {(time.time()-t0)/60:.1f} min")
else:
    print("graph metrics skipped")

  est. total 1.1 min for 11,444 windows (set COMPUTE_GRAPH_METRICS=False to skip)
graph metrics done in 0.5 min


## 9. Time-based split (no random edge split)
Windows are split **chronologically**, with `GAP_WINDOWS` purged before each boundary so an attack that straddles the boundary cannot leak.
- `chrono`: first 70 % of the timeline → train, next 15 % → val, last 15 % → test (concurrent windows from different datasets always land on the same side).
- `chrono_within_period`: same idea, applied inside each calendar-week block (all datasets together, so concurrent captures share boundaries), so every attack campaign appears in every split.

The report shows attack share and which techniques are **unseen in train** for both modes – pick the mode in `CFG.SPLIT_MODE`.

In [14]:
# [cell:splits]
win = con.execute("SELECT * FROM windows ORDER BY win_id").df()
ed_lab = con.execute("SELECT win_id, y, tech_mode FROM edges_win").df()

def assign_splits(win, mode, tr=CFG.TRAIN_FRAC, va=CFG.VAL_FRAC, gap=CFG.GAP_WINDOWS):
    ws = CFG.WINDOW_MIN * 60
    split = np.full(len(win), -1, dtype=np.int8)           # -1 = purged
    if mode == "chrono":
        groups = [np.arange(len(win))]
    elif mode == "chrono_within_period":
        wk = (win.win_start_ts // (7 * 86400)).astype(int)
        groups = [g.index.values for _, g in win.groupby(wk)]
    else:
        raise ValueError(mode)
    for idx in groups:
        t = win.win_start_ts.values[idx]; n = len(idx)
        if n < 10:                                          # tiny blocks: all train
            split[idx] = 0; continue
        ts_sorted = np.sort(t)
        ta = ts_sorted[int(n * tr)]; tb = ts_sorted[min(n - 1, int(n * (tr + va)))]
        seg = np.where(t < ta, 0, np.where(t < tb, 1, 2)).astype(np.int8)
        purge = ((seg == 0) & (t >= ta - gap * ws)) | ((seg == 1) & (t >= tb - gap * ws))
        seg[purge] = -1
        split[idx] = seg
    return split

def split_report(split, title):
    wsplit = pd.Series(split, index=win.win_id.values)
    e = ed_lab.assign(split=ed_lab.win_id.map(wsplit))
    rows, techs = [], {}
    for code_, name in [(0, "train"), (1, "val"), (2, "test"), (-1, "purged")]:
        s = e[e.split == code_]
        rows.append({"split": name, "windows": int((split == code_).sum()), "edges": len(s),
                     "attack_edges": int(s.y.sum()), "attack_%": round(100 * s.y.mean(), 1) if len(s) else 0})
        techs[name] = set(s.loc[s.y == 1, "tech_mode"].dropna())
    print(f"--- {title} ---"); print(pd.DataFrame(rows).to_string(index=False))
    for nm in ("val", "test"):
        unseen = sorted(techs[nm] - techs["train"])
        print(f"  techniques in {nm} but NOT in train: {unseen if unseen else 'none'}")

for mode in ("chrono", "chrono_within_period"):
    split_report(assign_splits(win, mode), mode)

win["split"] = assign_splits(win, CFG.SPLIT_MODE)
print("\nUSING:", CFG.SPLIT_MODE)
REPORT["split_mode"] = CFG.SPLIT_MODE

--- chrono ---
 split  windows  edges  attack_edges  attack_%
 train     8009 224685         51121      22.8
   val     1715 120340             0       0.0
  test     1717  93427           588       0.6
purged        3    231             0       0.0
  techniques in val but NOT in train: none
  techniques in test but NOT in train: none
--- chrono_within_period ---
 split  windows  edges  attack_edges  attack_%
 train     7981 302059         36177      12.0
   val     1684  66651          7736      11.6
  test     1723  68218          7520      11.0
purged       56   1755           276      15.7
  techniques in val but NOT in train: ['Credential Access_unspecified', 'T1571']
  techniques in test but NOT in train: ['Credential Access_unspecified', 'T1133', 'T1566']

USING: chrono_within_period


## 10. Feature matrices, train-only scaling, tensors
Count-like columns get `log1p`, then everything is standardised with **train-split statistics only** (the Phase A scaler was fitted on the whole graph).
Feature groups are named so ablations are one line (e.g. drop `volume_proxies` to test whether edge volume is a shortcut).

In [15]:
# [cell:pack]
import torch

edges = con.execute("SELECT * FROM edges_win ORDER BY win_id, src_id, dst_id").df()
edges["split"] = win.set_index("win_id").loc[edges.win_id.values, "split"].values
node_win["split"] = win.set_index("win_id").loc[node_win.win_id.values, "split"].values

EDGE_GROUPS = {
  "legacy":        ["edge_weight","avg_log_duration","avg_log_orig_bytes","avg_log_resp_bytes","total_tcp","total_udp","total_http","total_dns"],
  "proto_service": ["total_icmp","total_ssl","total_ssh","total_unknown","avg_log_orig_pkts","avg_log_resp_pkts"],
  "conn_state":    ["f_cs_sf","f_cs_s0","f_cs_rej","f_cs_rst","f_cs_oth","f_cs_partial"],
  "tcp_flags":     ["m_h_len","m_h_S","m_h_h","m_h_ack","m_h_data","m_h_fin","m_h_rst","m_h_retx"],
  "ports":         ["n_dst_ports","n_src_ports","f_pg_web","f_pg_ssh","f_pg_smb_rdp","f_pg_dns","f_pg_other_wk","f_pg_registered","f_pg_ephemeral"],
  "timing":        ["log_span","log_rate"],
}
EDGE_COLS = [c for g in EDGE_GROUPS.values() for c in g]
VOLUME_PROXIES = ["edge_weight","log_rate","log_span","total_tcp","total_udp","total_http","total_dns",
                  "total_icmp","total_ssl","total_ssh","total_unknown"]
LOG1P_EDGE = [c for c in EDGE_COLS if c.startswith("total_") or c in ("n_dst_ports","n_src_ports") or c.startswith("m_h_")]
NODE_COLS = ["out_deg_l","in_deg_l","out_logs_l","in_logs_l","out_lob","out_lrb","in_lob","in_lrb","out_fail","in_fail",
             "out_dur","in_dur","out_ports_l","in_ports_l","io_ratio","pagerank","clustering",
             "is_internal","is_ipv6","is_special"]
NODE_BINARY = ["is_internal","is_ipv6","is_special"]

def prep_edge(df):
    X = df[EDGE_COLS].astype("float64").copy()
    X[LOG1P_EDGE] = np.log1p(X[LOG1P_EDGE])
    return X

Xe = prep_edge(edges)
tr = (edges.split == 0).values
e_mu, e_sd = Xe[tr].mean(), Xe[tr].std().replace(0, 1.0).fillna(1.0)
Xe = ((Xe - e_mu) / e_sd).clip(-10, 10).astype("float32")

Xn = node_win[NODE_COLS].astype("float64").copy()
trn = (node_win.split == 0).values
cont = [c for c in NODE_COLS if c not in NODE_BINARY]
n_mu, n_sd = Xn.loc[trn, cont].mean(), Xn.loc[trn, cont].std().replace(0, 1.0).fillna(1.0)
Xn[cont] = ((Xn[cont] - n_mu) / n_sd).clip(-10, 10)
Xn = Xn.astype("float32")

# technique labels (MITRE head): only attack edges get an index; *_unspecified are kept but flagged
att_tech = edges.loc[edges.y == 1, "tech_mode"].dropna()
TECHS = sorted(att_tech.unique())
t2i = {t: i for i, t in enumerate(TECHS)}
y_tech = np.where(edges.y.values == 1, edges.tech_mode.map(t2i).fillna(-1).values, -1).astype("int16")
tech_specific = np.array([bool(re.match(r"^T\d{4}", t)) for t in TECHS])

# pointers so one window = one contiguous slice
n_win = len(win)
edge_ptr = np.searchsorted(edges.win_id.values, np.arange(n_win + 1)).astype("int64")
node_ptr = np.searchsorted(node_win.win_id.values, np.arange(n_win + 1)).astype("int64")
assert edge_ptr[-1] == len(edges) and node_ptr[-1] == len(node_win)

BUNDLE = {
  "edge_src": torch.as_tensor(edges.src_id.values.astype("int64")), "edge_dst": torch.as_tensor(edges.dst_id.values.astype("int64")),
  "x_edge": torch.as_tensor(Xe.values), "y": torch.as_tensor(edges.y.values.astype("int8")),
  "y_tech": torch.as_tensor(y_tech), "attack_frac": torch.as_tensor(edges.attack_frac.values.astype("float32")),
  "edge_split": torch.as_tensor(edges.split.values.astype("int8")),
  "node_gid": torch.as_tensor(node_win.node_id.values.astype("int64")), "x_node": torch.as_tensor(Xn.values),
  "edge_ptr": torch.as_tensor(edge_ptr), "node_ptr": torch.as_tensor(node_ptr),
  "win_split": torch.as_tensor(win.split.values.astype("int8")),
  "win_start_ts": torch.as_tensor(win.win_start_ts.values.astype("float64")),
  "meta": {"edge_cols": EDGE_COLS, "node_cols": NODE_COLS, "edge_groups": EDGE_GROUPS, "volume_proxies": VOLUME_PROXIES,
           "techniques": TECHS, "technique_is_specific": tech_specific.tolist(),
           "window_min": CFG.WINDOW_MIN, "split_mode": CFG.SPLIT_MODE, "label_rule": CFG.LABEL_RULE,
           "scaler": {"edge_mean": e_mu.to_dict(), "edge_std": e_sd.to_dict(), "node_mean": n_mu.to_dict(), "node_std": n_sd.to_dict(),
                      "log1p_edge_cols": LOG1P_EDGE}},
}
print(f"edges x_edge {tuple(Xe.shape)} | nodes x_node {tuple(Xn.shape)} | windows {n_win:,} | techniques {len(TECHS)}")
print("NaNs:", int(np.isnan(Xe.values).sum()), int(np.isnan(Xn.values).sum()))

edges x_edge (438683, 39) | nodes x_node (467856, 20) | windows 11,444 | techniques 33
NaNs: 0 0


## 11. Label-sparsity masks (100 / 15 / 10 / 5 / 1 % of attack labels)
`label_masks['p10'][e] = True` means edge *e* contributes to the training loss. All **normal** train edges stay labelled; attack train edges are kept at the given %,
**nested** (the 1 % set ⊂ 5 % ⊂ 10 % ⊂ 15 %) and **stratified by technique** (every technique keeps ≥ 1 edge). Val/test are always fully labelled.

*Team decision to confirm:* unlabelled attack edges are **masked out of the loss** (not treated as normal). Treating them as normal would add label noise; both are easy to switch.

In [16]:
# [cell:sparsity]
rng = np.random.default_rng(CFG.SEED)
y_np = edges.y.values; split_np = edges.split.values
is_tr_att = (split_np == 0) & (y_np == 1)

rank = np.ones(len(edges))                 # 0..1 within technique group; lowest ranks are kept first
tm = edges.tech_mode.fillna("NA").values
for t in np.unique(tm[is_tr_att]):
    idx = np.flatnonzero(is_tr_att & (tm == t))
    perm = rng.permutation(len(idx))
    rank[idx[perm]] = np.arange(len(idx)) / len(idx)     # first element -> 0.0 (kept at any p > 0)

label_masks, rows = {}, []
for p in CFG.SPARSITY_LEVELS:
    m = np.ones(len(edges), dtype=bool)
    m[is_tr_att] = rank[is_tr_att] < (p / 100.0) if p < 100 else True
    label_masks[f"p{p}"] = torch.as_tensor(m)
    kept = m & is_tr_att
    rows.append({"attack_label_%": p, "labelled_attack_edges": int(kept.sum()),
                 "labelled_normal_edges": int((m & (split_np == 0) & (y_np == 0)).sum()),
                 "techniques_covered": int(len(set(tm[kept])))})
BUNDLE["label_masks"] = label_masks
spars = pd.DataFrame(rows); print(spars.to_string(index=False))
REPORT["sparsity"] = spars.to_dict("records")

 attack_label_%  labelled_attack_edges  labelled_normal_edges  techniques_covered
            100                  36177                 265882                  28
             15                   5443                 265882                  28
             10                   3638                 265882                  28
              5                   1829                 265882                  28
              1                    383                 265882                  28


## 12. Window dataset class + save artifacts
`WindowGraphs` turns window *k* into a PyTorch-Geometric `Data` (active nodes re-indexed locally). It is written to `window_graphs.py` next to the data so the training notebooks can `import` it.

In [17]:
# [cell:window_class]
WINDOW_GRAPHS_SRC = """
import numpy as np, torch

class WindowGraphs:
    # One graph per time window. split: None | 0 (train) | 1 (val) | 2 (test)
    def __init__(self, path, split=None):
        self.d = torch.load(path, map_location="cpu", weights_only=False)
        ws = self.d["win_split"].numpy()
        self.win_ids = np.arange(len(ws)) if split is None else np.flatnonzero(ws == split)
        self.meta = self.d["meta"]

    def __len__(self):
        return len(self.win_ids)

    def get(self, k, mask_key=None):
        from torch_geometric.data import Data
        d = self.d
        e0, e1 = int(d["edge_ptr"][k]), int(d["edge_ptr"][k + 1])
        n0, n1 = int(d["node_ptr"][k]), int(d["node_ptr"][k + 1])
        gid = d["node_gid"][n0:n1].numpy()
        src = np.searchsorted(gid, d["edge_src"][e0:e1].numpy())
        dst = np.searchsorted(gid, d["edge_dst"][e0:e1].numpy())
        data = Data(x=d["x_node"][n0:n1],
                    edge_index=torch.as_tensor(np.stack([src, dst]), dtype=torch.long),
                    edge_attr=d["x_edge"][e0:e1], y=d["y"][e0:e1].float(),
                    y_tech=d["y_tech"][e0:e1].long(), node_gid=torch.as_tensor(gid),
                    win_id=torch.tensor([k]))
        if mask_key == "all":                     # attach every sparsity mask as m_p100, m_p15, ...
            for key, m in d["label_masks"].items():
                setattr(data, "m_" + key, m[e0:e1])
        elif mask_key is not None:
            data.label_mask = d["label_masks"][mask_key][e0:e1]
        return data

    def batch(self, ks, mask_key=None):
        from torch_geometric.data import Batch
        return Batch.from_data_list([self.get(int(k), mask_key) for k in ks])
"""
exec(WINDOW_GRAPHS_SRC)

In [18]:
# [cell:save]
out = CFG.OUT_DIR
torch.save(BUNDLE, out / f"uwf_windows_{CFG.WINDOW_MIN}min.pt")
(out / "window_graphs.py").write_text(WINDOW_GRAPHS_SRC)
edges.to_parquet(out / f"edges_win_{CFG.WINDOW_MIN}min.parquet", index=False)
node_win.to_parquet(out / f"node_win_{CFG.WINDOW_MIN}min.parquet", index=False)
win.to_parquet(out / f"windows_{CFG.WINDOW_MIN}min.parquet", index=False)
con.execute("SELECT * FROM node_static").df().to_csv(out / "node_static.csv", index=False)   # includes ip <-> node_id and subnet_id
json.dump(REPORT, open(out / "preprocessing_report.json", "w"), indent=1, default=str)
for f in sorted(out.iterdir()):
    print(f"{f.name:45s} {f.stat().st_size/1e6:8.1f} MB")

# quick sanity on the dataset class (needs torch_geometric)
try:
    ds = WindowGraphs(out / f"uwf_windows_{CFG.WINDOW_MIN}min.pt", split=0)
    g = ds.get(int(ds.win_ids[0]), mask_key="p1")
    print("example window graph:", g)
except Exception as e:
    print("WindowGraphs check skipped:", e)

edges_win_15min.parquet                           19.0 MB
node_static.csv                                    0.0 MB
node_win_15min.parquet                            10.1 MB
preprocessing_report.json                          0.0 MB
uwf_windows_15min.pt                             122.6 MB
window_graphs.py                                   0.0 MB
windows_15min.parquet                              0.2 MB
example window graph: Data(x=[43, 20], edge_index=[2, 40], edge_attr=[40, 39], y=[40], y_tech=[40], node_gid=[43], win_id=[1], label_mask=[40])


## 13. Viability summary
Plain-language checks on whether the graph is big/rich enough to learn from. Paste this output back.

In [19]:
# [cell:summary]
e = edges; sp = {0: "train", 1: "val", 2: "test"}
tr_att = int(((e.split == 0) & (e.y == 1)).sum())
checks = {
  "edges (samples)":                       f"{len(e):,}   (Phase A: 2,183)",
  "attack edges / normal edges":           f"{int(e.y.sum()):,} / {int((1-e.y).sum()):,}",
  "windows":                               f"{n_win:,}",
  "median active nodes per window":        int(node_win.groupby('win_id').size().median()),
  "median edges per window":               int(np.median(np.diff(edge_ptr))),
  "edges that mix attack+normal logs":     f"{100*((e.attack_frac>0)&(e.attack_frac<1)).mean():.2f}%",
  "train attack edges @100% / @1%":        f"{tr_att:,} / {int(label_masks['p1'].numpy()[(e.split==0)&(e.y==1)].sum()):,}" if 'p1' in label_masks else tr_att,
  "attack techniques present (distinct)":  len(TECHS),
  "techniques with a real T-ID":           int(tech_specific.sum()),
  "attack edges with only <Tactic>_unspecified": int(e[(e.y==1)].tech_mode.fillna('').str.endswith('_unspecified').sum()),
}
for k, v in checks.items():
    print(f"{k:48s} {v}")
REPORT["viability"] = {k: str(v) for k, v in checks.items()}
json.dump(REPORT, open(CFG.OUT_DIR / "preprocessing_report.json", "w"), indent=1, default=str)

edges (samples)                                  438,683   (Phase A: 2,183)
attack edges / normal edges                      51,709 / 386,974
windows                                          11,444
median active nodes per window                   39
median edges per window                          36
edges that mix attack+normal logs                0.00%
train attack edges @100% / @1%                   36,177 / 383
attack techniques present (distinct)             33
techniques with a real T-ID                      25
attack edges with only <Tactic>_unspecified      309


## 14. (Optional) Smoke test – is there signal, and is it a shortcut?
A deliberately small **edge-level GraphSAGE** (2 layers + edge MLP) trained on windows. It is *not* the final model; it answers:
1. Is there learnable signal on the time-based split (PR-AUC, F1 at a threshold chosen on **validation**, not test)?
2. Does performance collapse when **volume proxies** (`edge_weight`, rates, protocol totals) are removed? If yes, the model leans on traffic volume, not behaviour.
3. How does it degrade at 10 % and 1 % attack labels?

In [20]:
# [cell:smoke]
if not CFG.RUN_SMOKE_TEST:
    print("smoke test disabled (CFG.RUN_SMOKE_TEST=False)")
else:
    import torch.nn.functional as F
    from torch_geometric.nn import SAGEConv
    from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_fscore_support
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    exec_ns = {}; exec(WINDOW_GRAPHS_SRC, exec_ns)
    WG = exec_ns["WindowGraphs"]; P = out / f"uwf_windows_{CFG.WINDOW_MIN}min.pt"
    ds_all = {s: WG(P, split=s) for s in (0, 1, 2)}
    rs0 = np.random.default_rng(CFG.SEED)
    tr_ids = ds_all[0].win_ids
    if len(tr_ids) > CFG.SMOKE_MAX_TRAIN_WINDOWS:
        tr_ids = np.sort(rs0.choice(tr_ids, CFG.SMOKE_MAX_TRAIN_WINDOWS, replace=False))
    t0 = time.time()
    G = {0: [ds_all[0].get(int(k), "all") for k in tr_ids],
         1: [ds_all[1].get(int(k)) for k in ds_all[1].win_ids],
         2: [ds_all[2].get(int(k)) for k in ds_all[2].win_ids]}
    from torch_geometric.data import Batch
    print(f"cached window graphs in {time.time()-t0:.0f}s:", {k: len(v) for k, v in G.items()})
    ecols = BUNDLE["meta"]["edge_cols"]

    class EdgeSAGE(torch.nn.Module):
        def __init__(s, fn, fe, h=64):
            super().__init__()
            s.c1, s.c2 = SAGEConv(fn, h), SAGEConv(h, h)
            s.mlp = torch.nn.Sequential(torch.nn.Linear(2*h + fe, h), torch.nn.ReLU(), torch.nn.Dropout(0.2), torch.nn.Linear(h, 1))
        def forward(s, x, ei, ea):
            h = F.relu(s.c1(x, ei)); h = F.dropout(h, 0.2, s.training); h = F.relu(s.c2(h, ei))
            return s.mlp(torch.cat([h[ei[0]], h[ei[1]], ea], 1)).squeeze(-1)

    def run(p, drop=(), epochs=CFG.SMOKE_EPOCHS, bs=32):
        torch.manual_seed(CFG.SEED); rs = np.random.default_rng(CFG.SEED)
        keep = torch.tensor([i for i, c in enumerate(ecols) if c not in drop], device=dev)
        mk = f"p{p}"
        # pos_weight from labelled edges of the sampled train windows
        ys = torch.cat([g.y[getattr(g, "m_" + mk)] for g in G[0]])
        pos = int((ys == 1).sum()); neg = int((ys == 0).sum())
        model = EdgeSAGE(Xn.shape[1], len(keep)).to(dev)
        opt = torch.optim.Adam(model.parameters(), 1e-3, weight_decay=1e-4)
        pw = torch.tensor(neg / max(pos, 1), device=dev)
        def fwd(b): b = b.to(dev); return b, model(b.x, b.edge_index, b.edge_attr[:, keep])
        for ep in range(epochs):
            model.train(); ids = rs.permutation(len(G[0]))
            for i in range(0, len(ids), bs):
                b, lg = fwd(Batch.from_data_list([G[0][j] for j in ids[i:i+bs]]))
                lm = getattr(b, "m_" + mk)
                if lm.sum() == 0: continue
                loss = F.binary_cross_entropy_with_logits(lg[lm], b.y[lm], pos_weight=pw)
                opt.zero_grad(); loss.backward(); opt.step()
        @torch.no_grad()
        def scores(split):
            model.eval(); S, Y = [], []
            for i in range(0, len(G[split]), 64):
                b, lg = fwd(Batch.from_data_list(G[split][i:i+64]))
                S.append(torch.sigmoid(lg).cpu().numpy()); Y.append(b.y.cpu().numpy())
            return np.concatenate(S), np.concatenate(Y)
        sv, yv = scores(1); st, yt = scores(2)
        yv, yt = yv.astype(int), yt.astype(int)
        ths = np.linspace(0.05, 0.95, 19)
        f1s = [precision_recall_fscore_support(yv, sv >= t, average="binary", zero_division=0)[2] for t in ths]
        th = float(ths[int(np.argmax(f1s))])                     # threshold chosen on VALIDATION
        pr, rc, f1, _ = precision_recall_fscore_support(yt, st >= th, average="binary", zero_division=0)
        fpr = float(((st >= th) & (yt == 0)).sum() / max((yt == 0).sum(), 1))
        return {"attack_label_%": p, "dropped": "volume_proxies" if drop else "-", "threshold(val)": th,
                "test_PR_AUC": round(average_precision_score(yt, st), 4),
                "test_ROC_AUC": round(roc_auc_score(yt, st), 4) if len(set(yt)) > 1 else float("nan"),
                "test_precision": round(pr, 4), "test_recall": round(rc, 4), "test_F1": round(f1, 4), "test_FPR": round(fpr, 4),
                "test_attack_%": round(100 * yt.mean(), 1)}

    res = []
    for p, drop in [(100, ()), (100, VOLUME_PROXIES), (10, ()), (1, ())]:
        t0 = time.time(); r = run(p, drop); r["sec"] = round(time.time() - t0); res.append(r); print(r)
    smoke_df = pd.DataFrame(res); print(); print(smoke_df.to_string(index=False))
    REPORT["smoke_test"] = res
    json.dump(REPORT, open(CFG.OUT_DIR / "preprocessing_report.json", "w"), indent=1, default=str)

cached window graphs in 2s: {0: 4000, 1: 1684, 2: 1723}
{'attack_label_%': 100, 'dropped': '-', 'threshold(val)': 0.85, 'test_PR_AUC': np.float64(0.9995), 'test_ROC_AUC': np.float64(0.9999), 'test_precision': 0.9991, 'test_recall': 0.9968, 'test_F1': 0.9979, 'test_FPR': 0.0001, 'test_attack_%': np.float64(11.0), 'sec': 16}
{'attack_label_%': 100, 'dropped': 'volume_proxies', 'threshold(val)': 0.7999999999999999, 'test_PR_AUC': np.float64(0.9997), 'test_ROC_AUC': np.float64(0.9999), 'test_precision': 0.9948, 'test_recall': 0.9969, 'test_F1': 0.9959, 'test_FPR': 0.0006, 'test_attack_%': np.float64(11.0), 'sec': 14}
{'attack_label_%': 10, 'dropped': '-', 'threshold(val)': 0.7999999999999999, 'test_PR_AUC': np.float64(0.9992), 'test_ROC_AUC': np.float64(0.9997), 'test_precision': 0.9979, 'test_recall': 0.9964, 'test_F1': 0.9971, 'test_FPR': 0.0003, 'test_attack_%': np.float64(11.0), 'sec': 14}
{'attack_label_%': 1, 'dropped': '-', 'threshold(val)': 0.95, 'test_PR_AUC': np.float64(0.9979), 

In [21]:
from sklearn.metrics import roc_auc_score; te=(edges.split==2).values; print(pd.Series({c: roc_auc_score(edges.y[te], Xe[c][te]) for c in EDGE_COLS}).sub(.5).abs().add(.5).sort_values(ascending=False).head(10))

total_udp            0.969268
total_tcp            0.969004
total_dns            0.924235
m_h_S                0.923581
f_pg_dns             0.871620
total_unknown        0.847447
m_h_data             0.808108
avg_log_orig_pkts    0.788595
f_cs_s0              0.713473
m_h_len              0.704941
dtype: float64


In [22]:
from sklearn.neighbors import NearestNeighbors

hc_y, hc_sp, hc_X = edges.y.values, edges.split.values, Xe.values
tr_a = (hc_sp == 0) & (hc_y == 1)      # train attack edges
te_a = (hc_sp == 2) & (hc_y == 1)      # test attack edges

nn = NearestNeighbors(n_neighbors=2).fit(hc_X[tr_a])
d_te = nn.kneighbors(hc_X[te_a], n_neighbors=1)[0][:, 0]        # test -> nearest train attack edge
d_tr = nn.kneighbors(hc_X[tr_a], n_neighbors=2)[0][:, 1]        # train -> nearest OTHER train attack edge (reference)

qs = [0.1, 0.25, 0.5, 0.75, 0.9]
print("Distance to nearest train attack edge (smaller = more duplicated):")
print(pd.DataFrame({"test->train": np.quantile(d_te, qs),
                    "train->train (reference)": np.quantile(d_tr, qs)},
                   index=[f"p{int(q*100)}" for q in qs]).round(3))
for t in (0.05, 0.25, 1.0):
    print(f"  test attack edges with a train twin within {t}: {100*(d_te < t).mean():.1f}%")

pairs_tr = set(zip(edges.src_id.values[tr_a], edges.dst_id.values[tr_a]))
srcs_tr  = set(edges.src_id.values[tr_a])
pairs_te = list(zip(edges.src_id.values[te_a], edges.dst_id.values[te_a]))
print(f"\ntest attack edges whose (src->dst) pair also has attack edges in train: {100*np.mean([p in pairs_tr for p in pairs_te]):.1f}%")
print(f"test attack edges whose source IP is an attacker in train:               {100*np.mean([p[0] in srcs_tr for p in pairs_te]):.1f}%")
print(f"distinct attacker source IPs: train {len(srcs_tr)} | test {len(set(p[0] for p in pairs_te))} | in both {len(srcs_tr & set(p[0] for p in pairs_te))}")

Distance to nearest train attack edge (smaller = more duplicated):
     test->train  train->train (reference)
p10         0.00                      0.00
p25         0.00                      0.00
p50         0.00                      0.00
p75         0.00                      0.00
p90         0.01                      0.01
  test attack edges with a train twin within 0.05: 96.2%
  test attack edges with a train twin within 0.25: 98.5%
  test attack edges with a train twin within 1.0: 99.2%

test attack edges whose (src->dst) pair also has attack edges in train: 99.8%
test attack edges whose source IP is an attacker in train:               99.9%
distinct attacker source IPs: train 46 | test 31 | in both 26


In [23]:
import time, torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Batch
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_fscore_support

hc_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HC_EPOCHS, HC_CAP, HC_HELD_FRAC, HC_WIDE_GAP = 12, 4000, 0.25, 8     # 8 windows x 15 min = 2 h purge

_ns = {}; exec(WINDOW_GRAPHS_SRC, _ns)
hc_wg = _ns["WindowGraphs"](CFG.OUT_DIR / f"uwf_windows_{CFG.WINDOW_MIN}min.pt")

class HCEdgeSAGE(torch.nn.Module):
    def __init__(s, fn, fe, h=64):
        super().__init__()
        s.c1, s.c2 = SAGEConv(fn, h), SAGEConv(h, h)
        s.mlp = torch.nn.Sequential(torch.nn.Linear(2*h + fe, h), torch.nn.ReLU(), torch.nn.Dropout(0.2), torch.nn.Linear(h, 1))
    def forward(s, x, ei, ea):
        h = F.relu(s.c1(x, ei)); h = F.dropout(h, 0.2, s.training); h = F.relu(s.c2(h, ei))
        return s.mlp(torch.cat([h[ei[0]], h[ei[1]], ea], 1)).squeeze(-1)

def hc_fit(name, tr_ids, va_ids, te_ids, mk="p100", held=None, epochs=HC_EPOCHS, bs=32):
    torch.manual_seed(CFG.SEED); rs = np.random.default_rng(CFG.SEED)
    tr_ids = np.asarray(tr_ids)
    if len(tr_ids) > HC_CAP:
        tr_ids = np.sort(rs.choice(tr_ids, HC_CAP, replace=False))
    held_t = None if held is None else torch.as_tensor(np.asarray(held, dtype=np.int64))

    def build(ids):
        out = []
        for k in ids:
            g = hc_wg.get(int(k), "all")
            g.held = (torch.isin(g.node_gid[g.edge_index[0]], held_t) if held_t is not None
                      else torch.zeros(g.y.shape[0], dtype=torch.bool))
            out.append(g)
        return out
    Gtr, Gva, Gte = build(tr_ids), build(va_ids), build(te_ids)

    ys = torch.cat([g.y[getattr(g, "m_" + mk) & ~g.held] for g in Gtr])
    pos, neg = int((ys == 1).sum()), int((ys == 0).sum())
    if pos == 0 or neg == 0:
        return {"experiment": name, "note": "no labelled positives/negatives in train"}, None

    model = HCEdgeSAGE(Gtr[0].x.shape[1], Gtr[0].edge_attr.shape[1]).to(hc_dev)
    opt = torch.optim.Adam(model.parameters(), 1e-3, weight_decay=1e-4)
    pw = torch.tensor(neg / pos, device=hc_dev)
    for ep in range(epochs):
        model.train(); order = rs.permutation(len(Gtr))
        for i in range(0, len(order), bs):
            b = Batch.from_data_list([Gtr[j] for j in order[i:i+bs]]).to(hc_dev)
            lm = getattr(b, "m_" + mk) & ~b.held          # held-out source IPs never contribute to the loss
            if lm.sum() == 0: continue
            loss = F.binary_cross_entropy_with_logits(model(b.x, b.edge_index, b.edge_attr)[lm], b.y[lm], pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()

    @torch.no_grad()
    def scores(G, which):
        model.eval(); S, Y, T = [], [], []
        for i in range(0, len(G), 64):
            b = Batch.from_data_list(G[i:i+64]).to(hc_dev)
            if held is None:   sel = torch.ones_like(b.held)
            elif which == "te": sel = b.held                 # test only on held-out sources
            else:               sel = ~b.held                # validation only on seen sources
            s = torch.sigmoid(model(b.x, b.edge_index, b.edge_attr))
            S.append(s[sel].cpu().numpy()); Y.append(b.y[sel].cpu().numpy()); T.append(b.y_tech[sel].cpu().numpy())
        return np.concatenate(S), np.concatenate(Y).astype(int), np.concatenate(T)

    sv, yv, _ = scores(Gva, "va"); st, yt, tt = scores(Gte, "te")
    if len(set(yv)) < 2 or len(set(yt)) < 2:
        return {"experiment": name, "note": f"single class in val/test (val attacks {yv.sum()}, test attacks {yt.sum()})"}, None
    ths = np.linspace(0.05, 0.95, 19)                        # threshold chosen on VALIDATION
    th = float(ths[int(np.argmax([precision_recall_fscore_support(yv, sv >= t, average="binary", zero_division=0)[2] for t in ths]))])
    pr, rc, f1, _ = precision_recall_fscore_support(yt, st >= th, average="binary", zero_division=0)
    res = {"experiment": name, "labels_%": int(mk[1:]), "train_windows": len(Gtr), "test_edges": len(yt),
           "test_attack_%": round(100 * yt.mean(), 1), "thr(val)": round(th, 2),
           "PR_AUC": round(float(average_precision_score(yt, st)), 4), "ROC_AUC": round(float(roc_auc_score(yt, st)), 4),
           "precision": round(float(pr), 4), "recall": round(float(rc), 4), "F1": round(float(f1), 4),
           "FPR": round(float(((st >= th) & (yt == 0)).sum() / max((yt == 0).sum(), 1)), 4)}
    pt = pd.DataFrame({"tech": tt[yt == 1], "hit": st[yt == 1] >= th})
    pt["technique"] = pt.tech.map(lambda i: TECHS[i] if 0 <= i < len(TECHS) else "NA")
    return res, pt.groupby("technique").agg(test_edges=("hit", "size"), recall=("hit", "mean")).sort_values("test_edges", ascending=False)

hc_ids, hc_ws = win.win_id.values, win.split.values
hc_base = dict(tr=hc_ids[hc_ws == 0], va=hc_ids[hc_ws == 1], te=hc_ids[hc_ws == 2])
hc_rows, hc_per_tech = [], {}
def hc_go(name, d, mk="p100", held=None):
    t0 = time.time(); r, pt = hc_fit(name, d["tr"], d["va"], d["te"], mk, held)
    r["sec"] = round(time.time() - t0); hc_rows.append(r); hc_per_tech[(name, mk)] = pt; print(r)

# A) reference = the split you already used
hc_go("A within-period split (reference)", hc_base)
hc_go("A within-period split (reference)", hc_base, mk="p1")

# B) wide purge: 2 h gap around split boundaries (removes near-duplicate neighbours)
hc_sp8 = assign_splits(win, CFG.SPLIT_MODE, gap=HC_WIDE_GAP)
hc_go(f"B wide purge ({HC_WIDE_GAP*CFG.WINDOW_MIN} min gap)", dict(tr=hc_ids[hc_sp8 == 0], va=hc_ids[hc_sp8 == 1], te=hc_ids[hc_sp8 == 2]))

# C) unseen source IPs: 25% of IPs (attackers and non-attackers) never trained on, tested only on them
hc_rng = np.random.default_rng(CFG.SEED)
hc_att = np.unique(edges.src_id[edges.y == 1]); hc_nor = np.setdiff1d(np.unique(edges.src_id), hc_att)
hc_held = np.r_[hc_rng.choice(hc_att, max(1, int(HC_HELD_FRAC * len(hc_att))), replace=False),
                hc_rng.choice(hc_nor, int(HC_HELD_FRAC * len(hc_nor)), replace=False)]
print(f"attacker source IPs: {len(hc_att)} (held out {max(1, int(HC_HELD_FRAC*len(hc_att)))}) | other source IPs: {len(hc_nor)}")
hc_go("C unseen source IPs", hc_base, held=hc_held)

# D) different campaign: train on 2022 datasets, test on 2024-25 datasets
hc_old = win.source_dataset.isin(["UWF-ZeekData22", "UWF-ZeekDataFall22"]).values
hc_new = win.source_dataset.isin(["UWF-ZeekData24", "UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-1", "UWF-ZeekDataSum25-2"]).values
hc_d = dict(tr=hc_ids[hc_old & (hc_ws == 0)], va=hc_ids[hc_old & (hc_ws == 1)], te=hc_ids[hc_new])
hc_go("D train 2022 -> test 2024-25", hc_d)
hc_go("D train 2022 -> test 2024-25", hc_d, mk="p1")

hc_res = pd.DataFrame(hc_rows); print(); print(hc_res.to_string(index=False))
(CFG.OUT_DIR / "harder_checks_results.json").write_text(hc_res.to_json(orient="records", indent=1))

{'experiment': 'A within-period split (reference)', 'labels_%': 100, 'train_windows': 4000, 'test_edges': 68218, 'test_attack_%': np.float64(11.0), 'thr(val)': 0.9, 'PR_AUC': 0.9997, 'ROC_AUC': 1.0, 'precision': 0.9992, 'recall': 0.9965, 'F1': 0.9979, 'FPR': 0.0001, 'sec': 16}
{'experiment': 'A within-period split (reference)', 'labels_%': 1, 'train_windows': 4000, 'test_edges': 68218, 'test_attack_%': np.float64(11.0), 'thr(val)': 0.6, 'PR_AUC': 0.9977, 'ROC_AUC': 0.9991, 'precision': 0.9925, 'recall': 0.9918, 'F1': 0.9922, 'FPR': 0.0009, 'sec': 16}
{'experiment': 'B wide purge (120 min gap)', 'labels_%': 100, 'train_windows': 4000, 'test_edges': 68218, 'test_attack_%': np.float64(11.0), 'thr(val)': 0.8, 'PR_AUC': 0.9997, 'ROC_AUC': 1.0, 'precision': 0.9981, 'recall': 0.9969, 'F1': 0.9975, 'FPR': 0.0002, 'sec': 16}
attacker source IPs: 52 (held out 13) | other source IPs: 126
{'experiment': 'C unseen source IPs', 'labels_%': 100, 'train_windows': 4000, 'test_edges': 19676, 'test_attac

1684

In [24]:
for (name, mk), pt in hc_per_tech.items():
    if pt is None: continue
    print(f"\n--- per-technique recall | {name} | labels {mk[1:]}% ---")
    print(pt.round(3).head(15).to_string())


--- per-technique recall | A within-period split (reference) | labels 100% ---
                               test_edges  recall
technique                                        
T1595                                5509   0.998
T1110                                 766   1.000
T1078                                 623   1.000
T1190                                 554   0.998
T1592                                  32   0.781
T1048                                   8   1.000
Reconnaissance_unspecified              6   1.000
T1587                                   6   0.667
T1589                                   4   0.250
T1016                                   3   0.333
Credential Access_unspecified           2   1.000
T1210                                   2   0.500
Lateral Movement_unspecified            1   1.000
T1133                                   1   1.000
T1548                                   1   1.000

--- per-technique recall | A within-period split (reference) | labels

In [25]:
hc_e = edges.assign(dataset=win.source_dataset.values[edges.win_id.values])
print(hc_e.groupby("dataset").agg(edges=("y","size"), attack_edges=("y","sum")).assign(attack_pct=lambda d: (100*d.attack_edges/d.edges).round(1)))
print(pd.crosstab(hc_e.tech_mode[hc_e.y==1], hc_e.dataset[hc_e.y==1]).assign(total=lambda d: d.sum(axis=1)).sort_values("total", ascending=False).head(15))

                       edges  attack_edges  attack_pct
dataset                                               
UWF-ZeekData22         95863           309         0.3
UWF-ZeekData24         80481         49639        61.7
UWF-ZeekDataFall22     19058           322         1.7
UWF-ZeekDataFall24-2  100788           851         0.8
UWF-ZeekDataSum25-1    96586           315         0.3
UWF-ZeekDataSum25-2    45907           273         0.6
dataset                        UWF-ZeekData22  UWF-ZeekData24  UWF-ZeekDataFall22  UWF-ZeekDataFall24-2  UWF-ZeekDataSum25-1  UWF-ZeekDataSum25-2  total
tech_mode                                                                                                                                               
T1595                                       0           36406                  19                    32                  289                   15  36761
T1110                                       0            5186                   0                    

In [26]:
HC_CAP = 8000
hc_dsn = win.source_dataset.values
hc_others = ["UWF-ZeekData22", "UWF-ZeekDataFall22", "UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-1", "UWF-ZeekDataSum25-2"]
hc_rows_e = []
def hc_run(name, tr, va, te, mk="p100"):
    t0 = time.time(); r, _ = hc_fit(name, tr, va, te, mk); r["sec"] = round(time.time() - t0); hc_rows_e.append(r); print(r)

m24 = hc_dsn == "UWF-ZeekData24"
for X in hc_others:                                   # E: train on the big campaign only, test on each other dataset
    mx = hc_dsn == X
    for mk in ("p100", "p1"):
        hc_run(f"E train Data24 -> test {X[4:]}", hc_ids[m24 & (hc_ws == 0)], hc_ids[m24 & (hc_ws == 1)], hc_ids[mx], mk)

for X in hc_others + ["UWF-ZeekData24"]:              # F: leave-one-dataset-out
    mx = hc_dsn == X
    hc_run(f"F leave out {X[4:]}", hc_ids[~mx & (hc_ws == 0)], hc_ids[~mx & (hc_ws == 1)], hc_ids[mx])

hc_e_res = pd.DataFrame(hc_rows_e); hc_e_res["chance_PR_AUC"] = (hc_e_res["test_attack_%"] / 100).round(4)
cols = ["experiment", "labels_%", "test_edges", "test_attack_%", "chance_PR_AUC", "PR_AUC", "ROC_AUC", "precision", "recall", "F1", "FPR", "note"]
print(); print(hc_e_res[[c for c in cols if c in hc_e_res.columns]].to_string(index=False))
(CFG.OUT_DIR / "harder_checks_E_F.json").write_text(hc_e_res.to_json(orient="records", indent=1))

{'experiment': 'E train Data24 -> test ZeekData22', 'labels_%': 100, 'train_windows': 1512, 'test_edges': 95863, 'test_attack_%': np.float64(0.3), 'thr(val)': 0.2, 'PR_AUC': 0.8329, 'ROC_AUC': 0.8811, 'precision': 0.005, 'recall': 0.8608, 'F1': 0.01, 'FPR': 0.5531, 'sec': 19}
{'experiment': 'E train Data24 -> test ZeekData22', 'labels_%': 1, 'train_windows': 1512, 'test_edges': 95863, 'test_attack_%': np.float64(0.3), 'thr(val)': 0.3, 'PR_AUC': 0.82, 'ROC_AUC': 0.8753, 'precision': 0.0051, 'recall': 0.8544, 'F1': 0.01, 'FPR': 0.5441, 'sec': 8}
{'experiment': 'E train Data24 -> test ZeekDataFall22', 'labels_%': 100, 'train_windows': 1512, 'test_edges': 19058, 'test_attack_%': np.float64(1.7), 'thr(val)': 0.2, 'PR_AUC': 0.0574, 'ROC_AUC': 0.6995, 'precision': 0.0145, 'recall': 0.1056, 'F1': 0.0255, 'FPR': 0.1233, 'sec': 6}
{'experiment': 'E train Data24 -> test ZeekDataFall22', 'labels_%': 1, 'train_windows': 1512, 'test_edges': 19058, 'test_attack_%': np.float64(1.7), 'thr(val)': 0.3, '

4936

In [27]:
hc_ed, hc_yy, hc_tm, hc_XX = hc_e.dataset.values, hc_e.y.values, hc_e.tech_mode.values, Xe.values
def hc_shift(a, b, title):
    d = pd.Series(np.abs(hc_XX[a].mean(0) - hc_XX[b].mean(0)), index=EDGE_COLS).sort_values(ascending=False)
    print(f"\n{title}  (n={a.sum():,} vs {b.sum():,})"); print(d.head(8).round(2).to_string())
hc_shift((hc_ed == "UWF-ZeekData24") & (hc_yy == 0), (hc_ed != "UWF-ZeekData24") & (hc_yy == 0), "NORMAL edges: Data24 vs other datasets")
hc_shift((hc_ed == "UWF-ZeekData24") & (hc_tm == "T1595"), (hc_ed == "UWF-ZeekDataSum25-1") & (hc_tm == "T1595"), "T1595 attack edges: Data24 vs Sum25-1")


NORMAL edges: Data24 vs other datasets  (n=30,842 vs 356,132)
total_tcp             0.67
total_unknown         0.64
avg_log_orig_pkts     0.50
avg_log_orig_bytes    0.50
f_pg_dns              0.49
f_pg_other_wk         0.48
avg_log_duration      0.45
m_h_S                 0.38

T1595 attack edges: Data24 vs Sum25-1  (n=36,406 vs 289)
total_http       6.88
total_tcp        4.81
total_unknown    4.45
n_dst_ports      4.02
f_cs_partial     3.97
edge_weight      3.00
m_h_fin          2.94
n_src_ports      2.80


In [28]:
from sklearn.metrics import roc_auc_score
hc_tab = {}
for d in ["UWF-ZeekData24", "UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-1", "UWF-ZeekDataSum25-2", "UWF-ZeekDataFall22", "UWF-ZeekData22"]:
    m = hc_ed == d
    if hc_yy[m].min() == hc_yy[m].max(): continue
    hc_tab[d[4:]] = {c: roc_auc_score(hc_yy[m], hc_XX[m, i]) for i, c in enumerate(EDGE_COLS)}
hc_auc = pd.DataFrame(hc_tab); hc_auc["spread"] = hc_auc.max(axis=1) - hc_auc.min(axis=1)
print(hc_auc.sort_values("spread", ascending=False).head(12).round(2))

                    ZeekData24  ZeekDataFall24-2  ZeekDataSum25-1  ZeekDataSum25-2  ZeekDataFall22  ZeekData22  spread
m_h_len                   0.67              0.18             0.99             0.11            0.81        0.54    0.88
m_h_data                  0.20              0.04             0.69             0.02            0.74        0.10    0.72
avg_log_resp_pkts         0.69              0.32             0.99             0.50            0.86        0.51    0.67
avg_log_resp_bytes        0.49              0.32             0.96             0.47            0.87        0.49    0.63
f_cs_sf                   0.45              0.32             0.93             0.47            0.76        0.49    0.61
total_udp                 0.05              0.06             0.08             0.03            0.64        0.08    0.61
total_dns                 0.17              0.07             0.07             0.07            0.66        0.13    0.59
avg_log_orig_bytes        0.24              0.42

In [29]:
hc_rng2 = np.random.default_rng(CFG.SEED)
hc_keep = np.ones(len(edges), dtype=bool)
hc_idx = np.flatnonzero((hc_ed == "UWF-ZeekData24") & (hc_yy == 1))
hc_keep[hc_rng2.permutation(hc_idx)[:int(0.96 * len(hc_idx))]] = False       # keep ~4% of Data24's attack labels (~2,000 edges)
hc_wg.d["label_masks"]["p33"] = torch.as_tensor(hc_keep)                       # 'p33' = balanced mask (loss only; val/test untouched)
print("Data24 attack edges kept:", int((hc_keep & (hc_ed == "UWF-ZeekData24") & (hc_yy == 1)).sum()),
      "| attack edges from other datasets:", int(((hc_ed != "UWF-ZeekData24") & (hc_yy == 1)).sum()))

hc_rows_g = []
for X in hc_others + ["UWF-ZeekData24"]:
    mx = hc_dsn == X
    t0 = time.time(); r, _ = hc_fit(f"G balanced, leave out {X[4:]}", hc_ids[~mx & (hc_ws == 0)], hc_ids[~mx & (hc_ws == 1)], hc_ids[mx], "p33")
    r["sec"] = round(time.time() - t0); hc_rows_g.append(r)
hc_g = pd.DataFrame(hc_rows_g); hc_g["chance_PR_AUC"] = (hc_g["test_attack_%"] / 100).round(4)
cols = ["experiment", "test_attack_%", "chance_PR_AUC", "PR_AUC", "ROC_AUC", "precision", "recall", "F1", "FPR", "note"]
print(hc_g[[c for c in cols if c in hc_g.columns]].to_string(index=False))

Data24 attack edges kept: 1986 | attack edges from other datasets: 2070
                            experiment  test_attack_%  chance_PR_AUC  PR_AUC  ROC_AUC  precision  recall     F1    FPR
      G balanced, leave out ZeekData22            0.3          0.003  0.9274   0.9600     0.0501  0.9417 0.0952 0.0577
  G balanced, leave out ZeekDataFall22            1.7          0.017  0.6007   0.9737     0.8750  0.1304 0.2270 0.0003
G balanced, leave out ZeekDataFall24-2            0.8          0.008  0.0712   0.1957     0.0189  0.1210 0.0326 0.0536
 G balanced, leave out ZeekDataSum25-1            0.3          0.003  0.9011   0.9920     0.9920  0.7905 0.8799 0.0000
 G balanced, leave out ZeekDataSum25-2            0.6          0.006  0.9997   1.0000     0.9223  1.0000 0.9596 0.0005
      G balanced, leave out ZeekData24           61.7          0.617  0.9226   0.9046     0.9353  0.5646 0.7042 0.0629


In [30]:
for X in ["UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-1", "UWF-ZeekData22"]:
    mx = hc_dsn == X
    r, pt = hc_fit(f"leave out {X[4:]}", hc_ids[~mx & (hc_ws == 0)], hc_ids[~mx & (hc_ws == 1)], hc_ids[mx], "p33")
    print(f"\n{X}: ROC {r.get('ROC_AUC')}, F1 {r.get('F1')}"); print(pt.round(3).head(8).to_string() if pt is not None else r)


UWF-ZeekDataFall24-2: ROC 0.1598, F1 0.0339
           test_edges  recall
technique                    
T1046             274   0.091
T1018             273   0.084
T1548             254   0.020
T1595              32   1.000
T1016              10   1.000
T1210               5   1.000
T1190               2   1.000
T1071               1   1.000

UWF-ZeekDataSum25-1: ROC 0.9903, F1 0.8759
           test_edges  recall
technique                    
T1595             289   0.782
T1210               9   0.889
T1548               9   0.889
T1046               4   1.000
T1016               3   0.000
T1190               1   1.000

UWF-ZeekData22: ROC 0.9581, F1 0.0953
                                  test_edges  recall
technique                                           
Reconnaissance_unspecified               290   0.983
Credential Access_unspecified              6   0.000
Privilege Escalation_unspecified           3   0.333
Exfiltration_unspecified                   3   0.667
Lateral Moveme

In [31]:
hc_shift((hc_ed == "UWF-ZeekDataFall24-2") & (hc_tm == "T1046"), (hc_ed == "UWF-ZeekDataSum25-2") & (hc_tm == "T1046"), "T1046: Fall24-2 vs Sum25-2")
hc_shift((hc_ed == "UWF-ZeekDataFall24-2") & (hc_tm == "T1046"), (hc_ed == "UWF-ZeekDataFall24-2") & (hc_yy == 0), "Fall24-2: T1046 attack vs its own normal edges")


T1046: Fall24-2 vs Sum25-2  (n=274 vs 258)
total_icmp        2.54
log_rate          2.25
log_span          1.79
total_unknown     1.66
total_tcp         1.50
edge_weight       0.94
n_src_ports       0.72
f_pg_ephemeral    0.53

Fall24-2: T1046 attack vs its own normal edges  (n=274 vs 99,937)
total_icmp       5.33
n_dst_ports      4.54
total_unknown    4.19
total_tcp        3.63
f_pg_web         3.20
m_h_data         2.64
f_cs_oth         2.54
f_pg_dns         1.87


In [32]:
import copy

def hc_blocks(X, block=24, pool_frac=0.3, test_frac=0.5, min_att=60):
    ids = hc_ids[hc_dsn == X]; att = win.n_attack_edges.values[ids]; nb = int(np.ceil(len(ids) / block))
    for s in range(200):
        perm = np.random.default_rng(1000 + s).permutation(nb)
        role = np.full(nb, "gap", dtype=object)
        role[perm[:int(pool_frac * nb)]] = "pool"; role[perm[int(pool_frac * nb):int((pool_frac + test_frac) * nb)]] = "test"
        wr = np.repeat(role, block)[:len(ids)]
        if att[wr == "pool"].sum() >= min_att and att[wr == "test"].sum() >= min_att:
            return ids[wr == "pool"], ids[wr == "test"]
    return None, None

def hc_adapt(X, ks=(0, 5, 10, 25, 50), seeds=(0, 1, 2), pre_epochs=12, ft_epochs=6, bs=32):
    pool_ids, te_ids = hc_blocks(X)
    if pool_ids is None:
        print(X, "-> not enough attack edges for a pool/test split"); return []
    mx = hc_dsn == X; rs = np.random.default_rng(CFG.SEED); torch.manual_seed(CFG.SEED)
    tr_ids = hc_ids[~mx & (hc_ws == 0)]
    if len(tr_ids) > HC_CAP: tr_ids = np.sort(rs.choice(tr_ids, HC_CAP, replace=False))
    Gtr = [hc_wg.get(int(k), "all") for k in tr_ids]
    Gpool = [hc_wg.get(int(k), "all") for k in pool_ids]; Gte = [hc_wg.get(int(k), "all") for k in te_ids]
    ys = torch.cat([g.y for g in Gtr]); pw = torch.tensor(float((ys == 0).sum() / max((ys == 1).sum(), 1)), device=hc_dev)
    model = HCEdgeSAGE(Gtr[0].x.shape[1], Gtr[0].edge_attr.shape[1]).to(hc_dev)
    opt = torch.optim.Adam(model.parameters(), 1e-3, weight_decay=1e-4)
    for ep in range(pre_epochs):                                   # pretrain on the other datasets
        model.train(); order = rs.permutation(len(Gtr))
        for i in range(0, len(order), bs):
            b = Batch.from_data_list([Gtr[j] for j in order[i:i+bs]]).to(hc_dev)
            loss = F.binary_cross_entropy_with_logits(model(b.x, b.edge_index, b.edge_attr), b.y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()

    @torch.no_grad()
    def score(m, G):
        m.eval(); S, Y = [], []
        for i in range(0, len(G), 64):
            b = Batch.from_data_list(G[i:i+64]).to(hc_dev)
            S.append(torch.sigmoid(m(b.x, b.edge_index, b.edge_attr)).cpu().numpy()); Y.append(b.y.cpu().numpy())
        return np.concatenate(S), np.concatenate(Y).astype(int)

    att_pos = [(gi, ei) for gi, g in enumerate(Gpool) for ei in torch.nonzero(g.y == 1).flatten().tolist()]
    n_neg = sum(int((g.y == 0).sum()) for g in Gpool)
    out = []
    for k in ks:
        for sd in (seeds if k > 0 else (0,)):
            m = copy.deepcopy(model)
            if k > 0:
                r = np.random.default_rng(sd)
                for g in Gpool: g.lab = (g.y == 0).clone()          # all normals labelled, attacks unlabelled...
                for i in r.choice(len(att_pos), k, replace=False):  # ...except k labelled attack edges
                    gi, ei = att_pos[i]; Gpool[gi].lab[ei] = True
                o = torch.optim.Adam(m.parameters(), 3e-4, weight_decay=1e-4)
                w = torch.tensor(min(n_neg / k, 20.0), device=hc_dev)
                for ep in range(ft_epochs):
                    m.train(); order = r.permutation(len(Gpool))
                    for i in range(0, len(order), bs):
                        b = Batch.from_data_list([Gpool[j] for j in order[i:i+bs]]).to(hc_dev)
                        if b.lab.sum() == 0: continue
                        l = F.binary_cross_entropy_with_logits(m(b.x, b.edge_index, b.edge_attr)[b.lab], b.y[b.lab], pos_weight=w)
                        o.zero_grad(); l.backward(); o.step()
            sp_, yp_ = score(m, Gpool); st_, yt_ = score(m, Gte)
            thr = float(np.quantile(sp_[yp_ == 0], 0.99))           # 1% false-alarm budget on the target's own normal edges
            pred = st_ >= thr
            out.append({"target": X[4:], "k_attack_labels": k, "seed": sd, "pool_attack": len(att_pos), "test_attack": int(yt_.sum()),
                        "chance_PR": round(float(yt_.mean()), 4), "PR_AUC": round(float(average_precision_score(yt_, st_)), 4),
                        "ROC_AUC": round(float(roc_auc_score(yt_, st_)), 4),
                        "recall@1%FPRbudget": round(float(pred[yt_ == 1].mean()), 3), "test_FPR": round(float(pred[yt_ == 0].mean()), 4),
                        "precision": round(float(yt_[pred].mean()) if pred.any() else 0.0, 4)})
    return out

hc_fs = []
for X in ["UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-1", "UWF-ZeekDataFall22", "UWF-ZeekData22", "UWF-ZeekDataSum25-2"]:
    t0 = time.time(); hc_fs += hc_adapt(X); print(X, "done in", round(time.time() - t0), "s")
fs = pd.DataFrame(hc_fs)
agg = fs.groupby(["target", "k_attack_labels"]).agg(chance_PR=("chance_PR", "first"), PR_AUC=("PR_AUC", "mean"), ROC_AUC=("ROC_AUC", "mean"),
        recall=("recall@1%FPRbudget", "mean"), recall_sd=("recall@1%FPRbudget", "std"), test_FPR=("test_FPR", "mean")).round(3)
print(agg.to_string())
(CFG.OUT_DIR / "fewshot_adaptation.json").write_text(fs.to_json(orient="records", indent=1))

UWF-ZeekDataFall24-2 done in 43 s
UWF-ZeekDataSum25-1 done in 40 s
UWF-ZeekDataFall22 done in 41 s
UWF-ZeekData22 -> not enough attack edges for a pool/test split
UWF-ZeekData22 done in 0 s
UWF-ZeekDataSum25-2 -> not enough attack edges for a pool/test split
UWF-ZeekDataSum25-2 done in 0 s
                                  chance_PR  PR_AUC  ROC_AUC  recall  recall_sd  test_FPR
target           k_attack_labels                                                         
ZeekDataFall22   0                    0.017   0.514    0.947   0.682        NaN     0.016
                 5                    0.017   0.721    0.956   0.787      0.008     0.022
                 10                   0.017   0.740    0.953   0.777      0.000     0.020
                 25                   0.017   0.676    0.955   0.779      0.010     0.015
                 50                   0.017   0.674    0.960   0.788      0.010     0.018
ZeekDataFall24-2 0                    0.001   0.628    0.991   0.787        NaN

9603

In [33]:
def hc_blocks(X, block=24, pool_frac=0.3, test_frac=0.5, min_att=30, tol=0.12):
    ids = hc_ids[hc_dsn == X]; att = win.n_attack_edges.values[ids].astype(float); tot = att.sum()
    if tot == 0: return None, None
    nb = int(np.ceil(len(ids) / block))
    for s in range(3000):
        perm = np.random.default_rng(1000 + s).permutation(nb)
        role = np.full(nb, "gap", dtype=object)
        role[perm[:int(pool_frac * nb)]] = "pool"; role[perm[int(pool_frac * nb):int((pool_frac + test_frac) * nb)]] = "test"
        wr = np.repeat(role, block)[:len(ids)]
        pa, ta = att[wr == "pool"].sum(), att[wr == "test"].sum()
        if pa >= min_att and ta >= min_att and abs(pa / tot - pool_frac) <= tol and abs(ta / tot - test_frac) <= tol:
            return ids[wr == "pool"], ids[wr == "test"]
    return None, None

hc_fs = []
for X in ["UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-1", "UWF-ZeekDataFall22", "UWF-ZeekData22", "UWF-ZeekDataSum25-2"]:
    t0 = time.time(); hc_fs += hc_adapt(X); print(X, "done in", round(time.time() - t0), "s")
fs = pd.DataFrame(hc_fs)
agg = fs.groupby(["target", "k_attack_labels"]).agg(pool_att=("pool_attack", "first"), test_att=("test_attack", "first"), chance_PR=("chance_PR", "first"),
        PR_AUC=("PR_AUC", "mean"), ROC_AUC=("ROC_AUC", "mean"), recall=("recall@1%FPRbudget", "mean"), recall_sd=("recall@1%FPRbudget", "std"), test_FPR=("test_FPR", "mean")).round(3)
print(agg.to_string())
(CFG.OUT_DIR / "fewshot_adaptation.json").write_text(fs.to_json(orient="records", indent=1))

UWF-ZeekDataFall24-2 -> not enough attack edges for a pool/test split
UWF-ZeekDataFall24-2 done in 1 s
UWF-ZeekDataSum25-1 done in 44 s
UWF-ZeekDataFall22 done in 42 s
UWF-ZeekData22 -> not enough attack edges for a pool/test split
UWF-ZeekData22 done in 1 s
UWF-ZeekDataSum25-2 -> not enough attack edges for a pool/test split
UWF-ZeekDataSum25-2 done in 0 s
                                 pool_att  test_att  chance_PR  PR_AUC  ROC_AUC  recall  recall_sd  test_FPR
target          k_attack_labels                                                                             
ZeekDataFall22  0                     131       157      0.017   0.522    0.947   0.682        NaN     0.016
                5                     131       157      0.017   0.721    0.957   0.786      0.015     0.022
                10                    131       157      0.017   0.742    0.954   0.775      0.007     0.020
                25                    131       157      0.017   0.680    0.956   0.779      0.

6395

In [34]:
hc_blocks_strict = hc_blocks
hc_blocks = lambda X, **kw: (hc_ids[hc_dsn == X], hc_ids[hc_dsn == X])    # zero-shot only: calibrate on normal edges, test on the whole dataset
hc_zs = []
for X in ["UWF-ZeekDataFall24-2", "UWF-ZeekDataSum25-2", "UWF-ZeekData22", "UWF-ZeekDataSum25-1", "UWF-ZeekDataFall22"]:
    hc_zs += hc_adapt(X, ks=(0,))
hc_blocks = hc_blocks_strict
print(pd.DataFrame(hc_zs)[["target", "test_attack", "chance_PR", "PR_AUC", "ROC_AUC", "recall@1%FPRbudget", "test_FPR", "precision"]].to_string(index=False))

          target  test_attack  chance_PR  PR_AUC  ROC_AUC  recall@1%FPRbudget  test_FPR  precision
ZeekDataFall24-2          851     0.0084  0.0774   0.1627               0.088    0.0100     0.0696
 ZeekDataSum25-2          273     0.0059  0.9985   1.0000               1.000    0.0100     0.3740
      ZeekData22          309     0.0032  0.9279   0.9433               0.935    0.0101     0.2312
 ZeekDataSum25-1          315     0.0033  0.9074   0.9830               0.914    0.0100     0.2302
  ZeekDataFall22          322     0.0169  0.6058   0.9605               0.693    0.0101     0.5413


In [35]:
ip = con.execute("SELECT node_id, ip FROM node_static").df().set_index("node_id")["ip"]
att_wins = win[win.n_attack_edges > 0]
kw = att_wins.iloc[len(att_wins) // 2]                       # a mid-timeline window that contains attacks
k = int(kw.win_id)
ew = edges[edges.win_id == k].copy()
ew.insert(0, "src_ip", ew.src_id.map(ip)); ew.insert(1, "dst_ip", ew.dst_id.map(ip))
print(f"window {k} | dataset {kw.source_dataset} | starts {pd.to_datetime(kw.win_start_ts, unit='s')} UTC | {int(kw.n_edges)} edges, {int(kw.n_attack_edges)} attack")
cols = ["src_ip", "dst_ip", "n_logs", "y", "tech_mode", "n_dst_ports", "f_cs_s0", "f_cs_sf", "m_h_len", "avg_log_orig_bytes", "total_udp", "total_tcp", "total_dns"]
ew[cols].sort_values("y", ascending=False).head(15)

window 6179 | dataset UWF-ZeekData24 | starts 2024-03-16 23:30:00 UTC | 25 edges, 25 attack


,src_ip,dst_ip,n_logs,y,tech_mode,n_dst_ports,f_cs_s0,f_cs_sf,m_h_len,avg_log_orig_bytes,total_udp,total_tcp,total_dns
137721,143.88.11.10,143.88.12.14,1,1,T1078,1,1.0,0.000000,1.000000,0.000000,0.0,1.0,0.0
137722,143.88.12.12,143.88.13.10,40,1,T1110,1,0.0,1.000000,10.900000,6.526472,0.0,40.0,0.0
137723,143.88.13.12,143.88.14.10,1,1,T1190,1,0.0,1.000000,10.000000,6.429719,0.0,1.0,0.0
137724,143.88.13.12,143.88.14.12,250,1,T1110,1,0.0,1.000000,10.984000,6.709107,0.0,250.0,0.0
137725,143.88.14.11,143.88.15.1,2,1,T1595,1,1.0,0.000000,1.000000,0.000000,0.0,2.0,0.0
137726,143.88.14.11,143.88.15.10,1,1,T1595,1,0.0,0.000000,2.000000,0.000000,0.0,1.0,0.0
137727,143.88.14.11,143.88.15.11,1,1,T1595,1,0.0,0.000000,2.000000,0.000000,0.0,1.0,0.0
137728,143.88.14.11,143.88.15.12,1,1,T1078,1,1.0,0.000000,1.000000,0.000000,0.0,1.0,0.0
137729,143.88.14.11,143.88.15.14,3,1,T1595,2,0.0,0.666667,8.333333,3.858737,0.0,3.0,0.0
137730,143.88.15.10,143.88.1.19,1,1,T1078,1,1.0,0.000000,1.000000,0.000000,0.0,1.0,0.0


In [38]:
aw = win[win.n_attack_edges > 0].assign(attack_share=lambda d: d.n_attack_edges / d.n_edges)
print(aw.attack_share.describe().round(3)); print("windows with attacks that are 100% attack:", round(100 * (aw.attack_share == 1).mean(), 1), "%")

count    2179.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: attack_share, dtype: float64
windows with attacks that are 100% attack: 100.0 %


In [36]:
r = ew[ew.y == 1].sort_values("n_logs").iloc[0]
WSEC = CFG.WINDOW_MIN * 60
print("GROUPED ROW (what the model sees):"); print(r[cols].to_string())
raw = con.execute(f'''SELECT datetime, src_ip_zeek, dest_ip_zeek, dest_port_zeek, proto, service, conn_state, history, duration, orig_bytes, resp_bytes, label_tactic, label_technique
    FROM logs_clean WHERE src_ip_zeek='{r.src_ip}' AND dest_ip_zeek='{r.dst_ip}' AND source_dataset='{kw.source_dataset}'
    AND CAST(FLOOR(ts/{WSEC}) AS BIGINT) = {int(kw.w)} ORDER BY ts LIMIT 8''').df()
print(f"\nRAW LOGS behind it (first 8 of {int(r.n_logs)}):"); raw

GROUPED ROW (what the model sees):
src_ip                143.88.11.10
dst_ip                143.88.12.14
n_logs                           1
y                                1
tech_mode                    T1078
n_dst_ports                      1
f_cs_s0                        1.0
f_cs_sf                        0.0
m_h_len                        1.0
avg_log_orig_bytes             0.0
total_udp                      0.0
total_tcp                      1.0
total_dns                      0.0

RAW LOGS behind it (first 8 of 1):


,datetime,src_ip_zeek,dest_ip_zeek,dest_port_zeek,proto,service,conn_state,history,duration,orig_bytes,resp_bytes,label_tactic,label_technique
0,2024-03-16 23:36:56.525,143.88.11.10,143.88.12.14,445,tcp,unknown,S0,S,0.001114,0,0,Defense Evasion,T1078


In [37]:
print("edge matrix:", Xe.shape, "| node matrix:", Xn.shape)
print("\nStandardised features for 3 edges of that window (columns = edges):"); display(Xe.loc[ew.index[:3]].T.round(2).head(20))
ni = node_win.index[node_win.win_id == k][:5]
nodes_view = Xn.loc[ni].copy(); nodes_view.insert(0, "ip", node_win.loc[ni, "node_id"].map(ip).values)
print("\nNode features for 5 nodes in that window:"); nodes_view.round(2)

edge matrix: (438683, 39) | node matrix: (467856, 20)

Standardised features for 3 edges of that window (columns = edges):


,137721,137722,137723
edge_weight,-0.75,1.47,-0.75
avg_log_duration,-0.43,1.12,-0.33
avg_log_orig_bytes,-0.94,1.58,1.55
avg_log_resp_bytes,-0.45,4.04,2.14
total_tcp,0.57,4.39,0.57
total_udp,-1.06,-1.06,-1.06
total_http,-0.12,4.62,4.62
total_dns,-0.96,-0.96,-0.96
total_icmp,-0.17,-0.17,-0.17
total_ssl,-0.11,6.57,-0.11



Node features for 5 nodes in that window:


,ip,out_deg_l,in_deg_l,out_logs_l,in_logs_l,out_lob,out_lrb,in_lob,in_lrb,out_fail,in_fail,out_dur,in_dur,out_ports_l,in_ports_l,io_ratio,pagerank,clustering,is_internal,is_ipv6,is_special
167127,143.88.1.19,-0.83,0.42,-0.74,-0.21,-0.79,-0.40,-0.48,-0.30,-0.72,1.21,-0.40,-0.31,-0.92,0.75,-0.99,0.55,-0.02,1.0,0.0,0.0
167128,143.88.11.10,0.43,-0.90,-0.31,-0.67,-0.79,-0.40,-0.48,-0.30,1.42,-0.83,-0.39,-0.31,0.72,-0.97,1.03,-0.22,-0.02,1.0,0.0,0.0
167129,143.88.12.12,0.43,-0.90,1.56,-0.67,1.84,4.66,-0.48,-0.30,-0.72,-0.83,1.35,-0.31,0.72,-0.97,1.03,-0.22,-0.02,1.0,0.0,0.0
167130,143.88.12.14,-0.83,0.42,-0.74,-0.21,-0.79,-0.40,-0.48,-0.30,-0.72,1.21,-0.40,-0.31,-0.92,0.75,-0.99,0.44,-0.02,1.0,0.0,0.0
167131,143.88.13.10,-0.83,0.42,-0.74,1.80,-0.79,-0.40,2.59,5.38,-0.72,-0.83,-0.40,2.01,-0.92,0.75,-0.99,0.44,-0.02,1.0,0.0,0.0


In [39]:
import torch.nn as nn
from torch_geometric.data import Data

HC_CAP_MIX = 4000

class HCMLP(nn.Module):                                   # no-graph control: sees only the edge's own features
    def __init__(s, fe, h=64):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(fe, h), nn.ReLU(), nn.Dropout(0.2), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(s, x, ei, ea): return s.net(ea).squeeze(-1)

hc_nodes_of = {int(k): g.node_id.values for k, g in node_win.groupby("win_id")}
hc_win_ds = dict(zip(win.win_id.values.tolist(), win.source_dataset.values))
hc_win_att = dict(zip(win.win_id.values.tolist(), win.n_attack_edges.values))

def hc_plain(k): return hc_wg.get(int(k))

def hc_merge(gA, gN):                                     # union of two windows; shared IPs become one node
    gids = torch.unique(torch.cat([gA.node_gid, gN.node_gid]))
    iA, iN = torch.searchsorted(gids, gA.node_gid), torch.searchsorted(gids, gN.node_gid)
    x = torch.zeros(len(gids), gA.x.shape[1]); x[iN] = gN.x; x[iA] = gA.x   # shared nodes keep the attack window's features (approximation)
    return Data(x=x, edge_index=torch.cat([iA[gA.edge_index], iN[gN.edge_index]], 1),
                edge_attr=torch.cat([gA.edge_attr, gN.edge_attr]), y=torch.cat([gA.y, gN.y]),
                y_tech=torch.cat([gA.y_tech, gN.y_tech]), node_gid=gids, win_id=gA.win_id)

def hc_mixed_list(ids, seed):
    rs = np.random.default_rng(seed); ids = [int(k) for k in ids]
    att = [k for k in ids if hc_win_att[k] > 0]; by_ds = {}
    for k in ids:
        if hc_win_att[k] == 0: by_ds.setdefault(hc_win_ds[k], []).append(k)
    used, out, ov = set(), [], []
    for a in att:
        cands = by_ds.get(hc_win_ds[a], [])
        if not cands: continue
        pick = [int(c) for c in rs.choice(cands, min(30, len(cands)), replace=False)]
        free = [c for c in pick if c not in used] or pick
        best = max(free, key=lambda c: len(np.intersect1d(hc_nodes_of[a], hc_nodes_of[c])))
        used.add(best); ov.append(len(np.intersect1d(hc_nodes_of[a], hc_nodes_of[best])) / len(hc_nodes_of[a]))
        out.append(hc_merge(hc_plain(a), hc_plain(best)))
    return out, np.array(ov)

def hc_mix_share(G):                                      # share of attack edges that touch an IP which also has a normal edge
    hit = tot = 0
    for g in G:
        a, n = g.y == 1, g.y == 0
        if a.sum() == 0 or n.sum() == 0: continue
        nn_nodes = torch.unique(g.edge_index[:, n]); ei = g.edge_index[:, a]
        hit += int((torch.isin(ei[0], nn_nodes) | torch.isin(ei[1], nn_nodes)).sum()); tot += int(a.sum())
    return hit / max(tot, 1)

tr_ids, va_ids, te_ids = [[int(k) for k in hc_base[s]] for s in ("tr", "va", "te")]
rs0 = np.random.default_rng(CFG.SEED)
tr_att = [k for k in tr_ids if hc_win_att[k] > 0]; tr_nor = [k for k in tr_ids if hc_win_att[k] == 0]
tr_nor_s = [int(k) for k in rs0.choice(tr_nor, min(len(tr_nor), max(0, HC_CAP_MIX - len(tr_att))), replace=False)]

G_tr_orig = [hc_plain(k) for k in tr_att + tr_nor_s]
mix_tr, ov_tr = hc_mixed_list(tr_ids, 1); G_tr_mix = mix_tr + [hc_plain(k) for k in tr_nor_s]
G_va_orig = [hc_plain(k) for k in va_ids]; G_te_orig = [hc_plain(k) for k in te_ids]
G_va_mix, _ = hc_mixed_list(va_ids, 2); G_te_mix, ov_te = hc_mixed_list(te_ids, 3)
print(f"mixing check (test): attack windows merged {len(G_te_mix)} | median share of attack-window IPs also in the partner window {np.median(ov_te):.2f} | "
      f"attack edges touching an IP that also has a normal edge: mixed {100*hc_mix_share(G_te_mix):.1f}% vs original windows 0.0%")

def hc_train(G, kind, epochs=12, bs=32):
    torch.manual_seed(CFG.SEED); rs = np.random.default_rng(CFG.SEED)
    ys = torch.cat([g.y for g in G]); pw = torch.tensor(float((ys == 0).sum() / max((ys == 1).sum(), 1)), device=hc_dev)
    m = (HCEdgeSAGE(G[0].x.shape[1], G[0].edge_attr.shape[1]) if kind == "GNN" else HCMLP(G[0].edge_attr.shape[1])).to(hc_dev)
    opt = torch.optim.Adam(m.parameters(), 1e-3, weight_decay=1e-4)
    for ep in range(epochs):
        m.train(); order = rs.permutation(len(G))
        for i in range(0, len(order), bs):
            b = Batch.from_data_list([G[j] for j in order[i:i+bs]]).to(hc_dev)
            loss = F.binary_cross_entropy_with_logits(m(b.x, b.edge_index, b.edge_attr), b.y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
    return m

@torch.no_grad()
def hc_score(m, G):
    m.eval(); S, Y = [], []
    for i in range(0, len(G), 64):
        b = Batch.from_data_list(G[i:i+64]).to(hc_dev)
        S.append(torch.sigmoid(m(b.x, b.edge_index, b.edge_attr)).cpu().numpy()); Y.append(b.y.cpu().numpy())
    return np.concatenate(S), np.concatenate(Y).astype(int)

def hc_eval(m, Gva, Gte):
    sv, yv = hc_score(m, Gva); st, yt = hc_score(m, Gte)
    thr = float(np.quantile(sv[yv == 0], 0.99)); pred = st >= thr          # 1% false-alarm budget on validation normals
    return {"test_edges": len(yt), "attack_%": round(100 * yt.mean(), 1), "PR_AUC": round(float(average_precision_score(yt, st)), 4),
            "ROC_AUC": round(float(roc_auc_score(yt, st)), 4), "recall@1%FPR": round(float(pred[yt == 1].mean()), 3), "FPR": round(float(pred[yt == 0].mean()), 4)}

rows = []
for tr_name, Gtr in (("original", G_tr_orig), ("mixed", G_tr_mix)):
    for kind in ("GNN", "MLP"):
        t0 = time.time(); m = hc_train(Gtr, kind)
        for te_name, Gva, Gte in (("original", G_va_orig, G_te_orig), ("mixed", G_va_mix, G_te_mix)):
            rows.append({"train_on": tr_name, "test_on": te_name, "model": kind, **hc_eval(m, Gva, Gte)})
        print(f"trained {kind} on {tr_name} windows ({time.time()-t0:.0f}s)")
hc_mixres = pd.DataFrame(rows).sort_values(["test_on", "train_on", "model"]); print(); print(hc_mixres.to_string(index=False))
(CFG.OUT_DIR / "mixed_window_test.json").write_text(hc_mixres.to_json(orient="records", indent=1))

mixing check (test): attack windows merged 325 | median share of attack-window IPs also in the partner window 0.50 | attack edges touching an IP that also has a normal edge: mixed 69.6% vs original windows 0.0%
trained GNN on original windows (12s)
trained MLP on original windows (9s)
trained GNN on mixed windows (12s)
trained MLP on mixed windows (8s)

train_on  test_on model  test_edges  attack_%  PR_AUC  ROC_AUC  recall@1%FPR    FPR
   mixed    mixed   GNN       28198      26.7  0.9919   0.9979         0.995 0.0139
   mixed    mixed   MLP       28198      26.7  0.9891   0.9966         0.811 0.0043
original    mixed   GNN       28198      26.7  0.8785   0.9641         0.376 0.0103
original    mixed   MLP       28198      26.7  0.9735   0.9932         0.672 0.0064
   mixed original   GNN       68218      11.0  0.9980   0.9998         0.999 0.0108
   mixed original   MLP       68218      11.0  0.9981   0.9996         0.999 0.0104
original original   GNN       68218      11.0  0.9997   

1510